# PathoPredictor — Reproducible Reconstruction

**Reconstructed analysis, not an exact replication of the published study.**

This workflow uses updated ClinVar/dbNSFP annotations, 203,103 annotation records, 201,225 unique variants, and 13 predictors.

The published article reported 59,302 variants and 28 predictors. Its original processed dataset has not been verified as recovered.

Run cells sequentially from a fresh Python kernel.

## 1. Load the reconstructed annotation dataset

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# Resolve repository root from the notebook's working directory.
# Supported launch locations:
#   repository root
#   repository/notebooks
cwd = Path.cwd().resolve()

if (cwd / "data" / "pathopredictor_matched_missense_raw.csv").is_file():
    REPO_ROOT = cwd
elif (
    cwd.parent / "data" / "pathopredictor_matched_missense_raw.csv"
).is_file():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Run Jupyter from the repository root or its notebooks folder."
    )

DATA_DIR = REPO_ROOT / "data"

source_file = DATA_DIR / "pathopredictor_matched_missense_raw.csv"

df = pd.read_csv(
    source_file,
    dtype={
        "chrom": "string",
        "pos": "string",
        "ref": "string",
        "alt": "string",
        "label": "Int64"
    },
    low_memory=False
)

assert len(df) == 203103, (
    f"Unexpected annotation record count: {len(df)}"
)

print("Repository:", REPO_ROOT)
print("Annotation records:", len(df))
print("Columns:", len(df.columns))


# Repository output locations
RESULTS_DIR = REPO_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
MODELS_DIR = REPO_ROOT / "models"

for directory in (TABLES_DIR, FIGURES_DIR, MODELS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Tables:", TABLES_DIR)
print("Figures:", FIGURES_DIR)
print("Models:", MODELS_DIR)


## Analysis Cell 39

In [ ]:
# ============================================================
# Cell 39 — Duplicate-Resolution Strategy Audit (Corrected)
# ============================================================

import pandas as pd
import numpy as np

print("=== Cell 39: Duplicate-Resolution Strategy Audit ===")

variant_keys = ["chrom", "pos", "ref", "alt"]

predictors_39 = [
    "SIFT_score",
    "Polyphen2_HDIV_score",
    "Polyphen2_HVAR_score",
    "MutationTaster_score",
    "MutationAssessor_score",
    "PROVEAN_score",
    "VEST4_score",
    "MetaSVM_score",
    "MetaLR_score",
    "REVEL_score",
    "CADD_phred",
    "DANN_score",
    "fathmm-XF_coding_score"
]

transcript_predictors_39 = {
    "SIFT_score",
    "Polyphen2_HDIV_score",
    "Polyphen2_HVAR_score",
    "MutationTaster_score",
    "MutationAssessor_score",
    "PROVEAN_score",
    "VEST4_score",
    "REVEL_score"
}

def numeric_score_39(value, first_only=False):
    if pd.isna(value):
        return np.nan

    parts = str(value).split(";")

    if first_only:
        parts = parts[:1]

    for part in parts:
        try:
            number = float(part)
            if np.isfinite(number):
                return number
        except (ValueError, TypeError):
            continue

    return np.nan


# ------------------------------------------------------------
# 1. Build predictor availability audit table
# ------------------------------------------------------------

audit_39 = df[variant_keys + ["label"]].copy()

for predictor in predictors_39:

    audit_39[f"historical_{predictor}"] = (
        df[predictor].map(
            lambda x, p=predictor: numeric_score_39(
                x,
                first_only=(p in transcript_predictors_39)
            )
        )
    )

    audit_39[f"available_{predictor}"] = (
        df[predictor].map(
            lambda x: numeric_score_39(
                x,
                first_only=False
            )
        )
    )

historical_cols = [
    f"historical_{p}" for p in predictors_39
]

available_cols = [
    f"available_{p}" for p in predictors_39
]

audit_39["historical_score_count"] = (
    audit_39[historical_cols].notna().sum(axis=1)
)

audit_39["available_score_count"] = (
    audit_39[available_cols].notna().sum(axis=1)
)

# Preserve source row order for reproducible tie-breaking
audit_39["source_order"] = np.arange(len(audit_39))


# ------------------------------------------------------------
# 2. Apply three deterministic selection strategies
# ------------------------------------------------------------

strategies = {
    "First source record": (
        audit_39.drop_duplicates(
            subset=variant_keys,
            keep="first"
        )
    ),

    "Most historical scores": (
        audit_39.sort_values(
            ["historical_score_count", "source_order"],
            ascending=[False, True],
            kind="stable"
        ).drop_duplicates(
            subset=variant_keys,
            keep="first"
        )
    ),

    "Most first-available scores": (
        audit_39.sort_values(
            ["available_score_count", "source_order"],
            ascending=[False, True],
            kind="stable"
        ).drop_duplicates(
            subset=variant_keys,
            keep="first"
        )
    )
}


# ------------------------------------------------------------
# 3. Compare predictor completeness
# ------------------------------------------------------------

print("\n1. COMPARISON OF SELECTION RULES")

for name, selected in strategies.items():

    print(f"\n{name}")

    print("Unique variants:", len(selected))

    print(
        "Mean historical scores:",
        round(selected["historical_score_count"].mean(), 2)
    )

    print(
        "Mean first-available scores:",
        round(selected["available_score_count"].mean(), 2)
    )

    print(
        "Complete historical cases:",
        int((selected["historical_score_count"] == 13).sum())
    )

    print(
        "Complete first-available cases:",
        int((selected["available_score_count"] == 13).sum())
    )


# ------------------------------------------------------------
# 4. Correctly align genomic variant indexes before comparison
# ------------------------------------------------------------

print("\n2. SELECTION AGREEMENT")

baseline = (
    strategies["First source record"]
    .set_index(variant_keys)["source_order"]
    .sort_index()
)

for name in [
    "Most historical scores",
    "Most first-available scores"
]:

    alternative = (
        strategies[name]
        .set_index(variant_keys)["source_order"]
        .reindex(baseline.index)
    )

    changed = int(baseline.ne(alternative).sum())
    unchanged = len(baseline) - changed

    print(f"\n{name}")
    print(f"Same annotation selected: {unchanged:,}")
    print(f"Different annotation selected: {changed:,}")

    print(
        "Percentage changed:",
        f"{100 * changed / len(baseline):.3f}%"
    )


# ------------------------------------------------------------
# 5. Compare the two availability-based strategies
# ------------------------------------------------------------

historical = (
    strategies["Most historical scores"]
    .set_index(variant_keys)["source_order"]
    .sort_index()
)

available = (
    strategies["Most first-available scores"]
    .set_index(variant_keys)["source_order"]
    .reindex(historical.index)
)

changed_between_methods = int(
    historical.ne(available).sum()
)

print("\n3. HISTORICAL VS FIRST-AVAILABLE SELECTION")

print(
    "Different annotations selected:",
    f"{changed_between_methods:,}"
)

print(
    "Percentage changed:",
    f"{100 * changed_between_methods / len(historical):.3f}%"
)

print("\n=== Cell 39 Complete ===")

## Analysis Cell 40

In [ ]:
# ============================================================
# Cell 40 — Variant-Level Dataset Audit
# ============================================================

import pandas as pd
import numpy as np

print("=== Cell 40: Variant-Level Dataset Audit ===")

variant_keys = ["chrom", "pos", "ref", "alt"]

predictors_40 = [
    "SIFT_score",
    "Polyphen2_HDIV_score",
    "Polyphen2_HVAR_score",
    "MutationTaster_score",
    "MutationAssessor_score",
    "PROVEAN_score",
    "VEST4_score",
    "MetaSVM_score",
    "MetaLR_score",
    "REVEL_score",
    "CADD_phred",
    "DANN_score",
    "fathmm-XF_coding_score"
]

# Select exactly one annotation record per genomic variant.
selected_40 = (
    strategies["Most historical scores"]
    .sort_values("source_order")
    .copy()
)

historical_columns_40 = [
    f"historical_{p}" for p in predictors_40
]

# Build the numeric feature table without altering source data.
X_40 = selected_40[historical_columns_40].copy()
X_40.columns = predictors_40
X_40 = X_40.reset_index(drop=True)

y_40 = (
    selected_40["label"]
    .astype(int)
    .reset_index(drop=True)
)

variant_info_40 = (
    selected_40[variant_keys]
    .reset_index(drop=True)
)

# Verify one row per variant.
assert len(X_40) == 201225
assert len(X_40) == len(y_40)
assert not variant_info_40.duplicated().any()
assert y_40.isin([0, 1]).all()

print("\n1. DATASET DIMENSIONS")
print("Unique variants:", len(X_40))
print("Predictors:", X_40.shape[1])
print("Duplicate genomic variants:", int(variant_info_40.duplicated().sum()))

print("\n2. CLASS DISTRIBUTION")
class_counts_40 = y_40.value_counts().sort_index()

for label, count in class_counts_40.items():
    class_name = "Benign" if label == 0 else "Pathogenic"
    print(
        f"{class_name}: {count:,} "
        f"({100 * count / len(y_40):.2f}%)"
    )

print("\n3. PREDICTOR MISSINGNESS")
missing_40 = pd.DataFrame({
    "Predictor": predictors_40,
    "Missing count": X_40.isna().sum().values,
    "Missing %": (
        100 * X_40.isna().mean().values
    ).round(2)
})

print(
    missing_40
    .sort_values("Missing %", ascending=False)
    .to_string(index=False)
)

print("\n4. AVAILABLE PREDICTORS PER VARIANT")
available_count_40 = X_40.notna().sum(axis=1)

print(
    available_count_40
    .value_counts()
    .sort_index()
    .rename_axis("Available predictors")
    .to_string()
)

print("\n5. COMPLETE-CASE CLASS DISTRIBUTION")
complete_mask_40 = X_40.notna().all(axis=1)

print("Complete-case variants:", int(complete_mask_40.sum()))

complete_class_counts_40 = (
    y_40[complete_mask_40]
    .value_counts()
    .sort_index()
)

for label, count in complete_class_counts_40.items():
    class_name = "Benign" if label == 0 else "Pathogenic"
    print(f"{class_name}: {count:,}")

print("\n=== Cell 40 Complete ===")

## Analysis Cell 41

In [ ]:
# ============================================================
# Cell 41 — Predictor Missingness by Clinical Class
# ============================================================

import pandas as pd
import numpy as np

print("=== Cell 41: Missingness by Clinical Class ===")

# Use the variant-level data created in Cell 40.
assert len(X_40) == len(y_40) == 201225
assert y_40.isin([0, 1]).all()

benign_mask_41 = y_40 == 0
pathogenic_mask_41 = y_40 == 1

results_41 = []

for predictor in X_40.columns:

    benign_missing = X_40.loc[
        benign_mask_41, predictor
    ].isna().mean() * 100

    pathogenic_missing = X_40.loc[
        pathogenic_mask_41, predictor
    ].isna().mean() * 100

    results_41.append({
        "Predictor": predictor,
        "Benign missing %": round(benign_missing, 2),
        "Pathogenic missing %": round(pathogenic_missing, 2),
        "Difference (path - benign)": round(
            pathogenic_missing - benign_missing, 2
        )
    })

missingness_by_class_41 = pd.DataFrame(results_41)

print("\n1. MISSINGNESS BY CLASS")
print(
    missingness_by_class_41
    .sort_values(
        "Difference (path - benign)",
        key=lambda s: s.abs(),
        ascending=False
    )
    .to_string(index=False)
)

print("\n2. CLASS BALANCE: FULL VS COMPLETE-CASE DATA")

complete_mask_41 = X_40.notna().all(axis=1)

for name, mask in [
    ("Full dataset", pd.Series(True, index=X_40.index)),
    ("Complete cases", complete_mask_41),
    ("Incomplete cases", ~complete_mask_41)
]:
    labels = y_40.loc[mask]

    benign_count = int((labels == 0).sum())
    pathogenic_count = int((labels == 1).sum())

    print(f"\n{name}")
    print(f"Total: {len(labels):,}")
    print(
        f"Benign: {benign_count:,} "
        f"({100 * benign_count / len(labels):.2f}%)"
    )
    print(
        f"Pathogenic: {pathogenic_count:,} "
        f"({100 * pathogenic_count / len(labels):.2f}%)"
    )

print("\n3. MEAN AVAILABLE PREDICTORS BY CLASS")

available_scores_41 = X_40.notna().sum(axis=1)

for label, name in [(0, "Benign"), (1, "Pathogenic")]:
    mean_available = available_scores_41[
        y_40 == label
    ].mean()

    print(f"{name}: {mean_available:.2f} of 13")

print("\n=== Cell 41 Complete ===")

## Analysis Cell 42

In [ ]:
# ============================================================
# Cell 42 — Stratified Train/Test Split and Leakage Audit
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("=== Cell 42: Stratified Train/Test Split ===")

# Confirm that features, labels, and genomic identifiers align.
assert len(X_40) == len(y_40) == len(variant_info_40)
assert X_40.index.equals(y_40.index)
assert X_40.index.equals(variant_info_40.index)
assert not variant_info_40.duplicated().any()

# Split row indices so genomic identifiers remain aligned.
all_indices_42 = np.arange(len(y_40))

train_idx_42, test_idx_42 = train_test_split(
    all_indices_42,
    test_size=0.20,
    random_state=42,
    stratify=y_40
)

# Create independent training and test datasets.
X_train_42 = X_40.iloc[train_idx_42].reset_index(drop=True)
X_test_42 = X_40.iloc[test_idx_42].reset_index(drop=True)

y_train_42 = y_40.iloc[train_idx_42].reset_index(drop=True)
y_test_42 = y_40.iloc[test_idx_42].reset_index(drop=True)

variants_train_42 = (
    variant_info_40.iloc[train_idx_42].reset_index(drop=True)
)

variants_test_42 = (
    variant_info_40.iloc[test_idx_42].reset_index(drop=True)
)

print("\n1. SPLIT DIMENSIONS")
print(f"Training variants: {len(X_train_42):,}")
print(f"Test variants:     {len(X_test_42):,}")
print(f"Predictors:        {X_train_42.shape[1]}")

print("\n2. CLASS DISTRIBUTION")

for name, labels in [
    ("Training", y_train_42),
    ("Test", y_test_42)
]:
    benign = int((labels == 0).sum())
    pathogenic = int((labels == 1).sum())

    print(f"\n{name} set")
    print(f"Benign: {benign:,} ({100 * benign / len(labels):.2f}%)")
    print(
        f"Pathogenic: {pathogenic:,} "
        f"({100 * pathogenic / len(labels):.2f}%)"
    )

print("\n3. GENOMIC VARIANT OVERLAP AUDIT")

train_variant_keys_42 = set(
    map(tuple, variants_train_42.to_numpy())
)

test_variant_keys_42 = set(
    map(tuple, variants_test_42.to_numpy())
)

overlap_42 = train_variant_keys_42.intersection(
    test_variant_keys_42
)

print(f"Overlapping genomic variants: {len(overlap_42):,}")

assert len(overlap_42) == 0, (
    "Data leakage detected: variants overlap across sets."
)

print("\n4. MISSINGNESS BEFORE IMPUTATION")
print(
    f"Training missing values: "
    f"{int(X_train_42.isna().sum().sum()):,}"
)
print(
    f"Test missing values:     "
    f"{int(X_test_42.isna().sum().sum()):,}"
)

print("\n5. DATA INTEGRITY CHECKS")

assert len(X_train_42) + len(X_test_42) == len(X_40)
assert list(X_train_42.columns) == list(X_test_42.columns)
assert y_train_42.isin([0, 1]).all()
assert y_test_42.isin([0, 1]).all()

print("All split integrity checks passed.")
print("\n=== Cell 42 Complete ===")

## Analysis Cell 43

In [ ]:
# ============================================================
# Cell 43 — Training-Only Median Imputation
# ============================================================

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

print("=== Cell 43: Training-Only Median Imputation ===")

# ------------------------------------------------------------
# 1. Confirm training and test data integrity
# ------------------------------------------------------------

assert len(X_train_42) == 160980
assert len(X_test_42) == 40245
assert X_train_42.shape[1] == 13
assert X_train_42.columns.equals(X_test_42.columns)

feature_names_43 = X_train_42.columns.tolist()

# ------------------------------------------------------------
# 2. Fit imputer on training data ONLY
# ------------------------------------------------------------

imputer_43 = SimpleImputer(
    missing_values=np.nan,
    strategy="median"
)

X_train_imputed_array_43 = imputer_43.fit_transform(
    X_train_42
)

# Transform test data using TRAINING medians
X_test_imputed_array_43 = imputer_43.transform(
    X_test_42
)

# ------------------------------------------------------------
# 3. Restore feature names and DataFrame structure
# ------------------------------------------------------------

assert X_train_imputed_array_43.shape[1] == len(feature_names_43), (
    "A predictor column was dropped during imputation."
)

assert X_test_imputed_array_43.shape[1] == len(feature_names_43), (
    "A predictor column was dropped during imputation."
)

X_train_imputed_43 = pd.DataFrame(
    X_train_imputed_array_43,
    columns=feature_names_43
)

X_test_imputed_43 = pd.DataFrame(
    X_test_imputed_array_43,
    columns=feature_names_43
)

# ------------------------------------------------------------
# 4. Report training-derived medians
# ------------------------------------------------------------

median_table_43 = pd.DataFrame({
    "Predictor": feature_names_43,
    "Training median": imputer_43.statistics_,
    "Training missing": X_train_42.isna().sum().values,
    "Test missing": X_test_42.isna().sum().values
})

print("\n1. TRAINING-DERIVED IMPUTATION VALUES")
print(median_table_43.to_string(index=False))

# ------------------------------------------------------------
# 5. Verify imputation
# ------------------------------------------------------------

print("\n2. IMPUTED DATASET DIMENSIONS")
print("Training:", X_train_imputed_43.shape)
print("Test:    ", X_test_imputed_43.shape)

print("\n3. MISSING VALUES AFTER IMPUTATION")

train_missing_43 = int(
    X_train_imputed_43.isna().sum().sum()
)

test_missing_43 = int(
    X_test_imputed_43.isna().sum().sum()
)

print("Training missing values:", train_missing_43)
print("Test missing values:    ", test_missing_43)

assert train_missing_43 == 0
assert test_missing_43 == 0

assert np.isfinite(X_train_imputed_array_43).all()
assert np.isfinite(X_test_imputed_array_43).all()

print("\n4. LEAKAGE SAFEGUARD")
print("Imputer fitted on training data only.")
print("Test data transformed using training-derived medians.")
print("No test-set statistics used for imputation.")

print("\n=== Cell 43 Complete ===")

## Analysis Cell 44

In [ ]:
# ============================================================
# Cell 44 — Training-Only Feature Scaling
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

print("=== Cell 44: Training-Only Feature Scaling ===")

# ------------------------------------------------------------
# 1. Verify that Cell 43 data is available
# ------------------------------------------------------------

required_variables_44 = [
    "X_train_imputed_43",
    "X_test_imputed_43",
    "y_train_42",
    "y_test_42"
]

missing_variables_44 = [
    name for name in required_variables_44
    if name not in globals()
]

if missing_variables_44:
    raise RuntimeError(
        "Please rerun the necessary earlier notebook cells "
        "before Cell 44. Missing variables: "
        + ", ".join(missing_variables_44)
    )

assert X_train_imputed_43.shape == (160980, 13)
assert X_test_imputed_43.shape == (40245, 13)

assert X_train_imputed_43.columns.equals(
    X_test_imputed_43.columns
)

assert len(y_train_42) == len(X_train_imputed_43)
assert len(y_test_42) == len(X_test_imputed_43)

feature_names_44 = X_train_imputed_43.columns.tolist()

# ------------------------------------------------------------
# 2. Fit StandardScaler on training data ONLY
# ------------------------------------------------------------

scaler_44 = StandardScaler()

X_train_scaled_array_44 = scaler_44.fit_transform(
    X_train_imputed_43
)

# Apply training-derived scaling parameters to test data
X_test_scaled_array_44 = scaler_44.transform(
    X_test_imputed_43
)

# ------------------------------------------------------------
# 3. Restore DataFrame structure
# ------------------------------------------------------------

X_train_scaled_44 = pd.DataFrame(
    X_train_scaled_array_44,
    columns=feature_names_44
)

X_test_scaled_44 = pd.DataFrame(
    X_test_scaled_array_44,
    columns=feature_names_44
)

# ------------------------------------------------------------
# 4. Report training-derived scaling parameters
# ------------------------------------------------------------

scaling_table_44 = pd.DataFrame({
    "Predictor": feature_names_44,
    "Training mean": scaler_44.mean_,
    "Training standard deviation": scaler_44.scale_
})

print("\n1. TRAINING-DERIVED SCALING PARAMETERS")
print(
    scaling_table_44.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ------------------------------------------------------------
# 5. Verify scaled datasets
# ------------------------------------------------------------

print("\n2. SCALED DATASET DIMENSIONS")
print("Training:", X_train_scaled_44.shape)
print("Test:    ", X_test_scaled_44.shape)

print("\n3. SCALED TRAINING DATA CHECKS")

training_means_44 = X_train_scaled_44.mean()
training_std_44 = X_train_scaled_44.std(ddof=0)

print(
    "Maximum absolute training mean:",
    f"{training_means_44.abs().max():.10f}"
)

print(
    "Maximum deviation from unit standard deviation:",
    f"{(training_std_44 - 1).abs().max():.10f}"
)

# ------------------------------------------------------------
# 6. Integrity and leakage safeguards
# ------------------------------------------------------------

assert np.isfinite(X_train_scaled_array_44).all()
assert np.isfinite(X_test_scaled_array_44).all()

assert np.allclose(
    training_means_44.to_numpy(),
    0,
    atol=1e-8
)

assert np.allclose(
    training_std_44.to_numpy(),
    1,
    atol=1e-8
)

print("\n4. LEAKAGE SAFEGUARD")
print("Scaler fitted on training data only.")
print("Test data transformed using training-derived parameters.")
print("No test-set statistics used for scaling.")

print("\nAll scaling integrity checks passed.")
print("\n=== Cell 44 Complete ===")

## Analysis Cell 45

In [ ]:
# ============================================================
# Cell 45 — Logistic Regression Baseline
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== Cell 45: Logistic Regression Baseline ===")

# 1. Verify inputs
assert X_train_scaled_44.shape == (160980, 13)
assert X_test_scaled_44.shape == (40245, 13)
assert len(y_train_42) == len(X_train_scaled_44)
assert len(y_test_42) == len(X_test_scaled_44)
assert np.isfinite(X_train_scaled_44.to_numpy()).all()
assert np.isfinite(X_test_scaled_44.to_numpy()).all()

# 2. Fit on training data only
logreg_45 = LogisticRegression(
    solver="lbfgs",
    C=1.0,
    max_iter=2000,
    random_state=42
)

logreg_45.fit(X_train_scaled_44, y_train_42)

# 3. Evaluate on the held-out test set
y_pred_45 = logreg_45.predict(X_test_scaled_44)
y_prob_45 = logreg_45.predict_proba(X_test_scaled_44)[:, 1]

results_45 = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test_42, y_pred_45),
    "Balanced accuracy": balanced_accuracy_score(y_test_42, y_pred_45),
    "Precision": precision_score(y_test_42, y_pred_45),
    "Recall (sensitivity)": recall_score(y_test_42, y_pred_45),
    "Specificity": recall_score(
        y_test_42, y_pred_45, pos_label=0
    ),
    "F1": f1_score(y_test_42, y_pred_45),
    "ROC-AUC": roc_auc_score(y_test_42, y_prob_45),
    "PR-AUC": average_precision_score(y_test_42, y_prob_45)
}

print("\n1. HELD-OUT TEST METRICS")
for metric, value in results_45.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

print("\n2. CONFUSION MATRIX")
cm_45 = confusion_matrix(
    y_test_42,
    y_pred_45,
    labels=[0, 1]
)

print(
    pd.DataFrame(
        cm_45,
        index=["Actual benign", "Actual pathogenic"],
        columns=["Predicted benign", "Predicted pathogenic"]
    ).to_string()
)

print("\n3. MODEL CONVERGENCE")
print("Iterations used:", int(logreg_45.n_iter_[0]))
print("Maximum iterations:", logreg_45.max_iter)

print("\n4. REPRODUCIBILITY")
print("Training variants:", len(y_train_42))
print("Test variants:", len(y_test_42))
print("Predictors:", X_train_scaled_44.shape[1])
print("Classification threshold: 0.50")
print("Test data excluded from model fitting.")

print("\n=== Cell 45 Complete ===")

## Analysis Cell 46

In [ ]:
# ============================================================
# Cell 46 — Training-Only Validation Split
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

print("=== Cell 46: Training-Only Validation Split ===")

# ------------------------------------------------------------
# 1. Verify original training and test partitions
# ------------------------------------------------------------

assert len(X_train_42) == len(y_train_42) == 160980
assert len(X_test_42) == len(y_test_42) == 40245

assert len(variants_train_42) == len(X_train_42)
assert len(variants_test_42) == len(X_test_42)

# ------------------------------------------------------------
# 2. Split original training rows: 80% fit, 20% validation
# ------------------------------------------------------------

fit_idx_46, val_idx_46 = train_test_split(
    np.arange(len(y_train_42)),
    test_size=0.20,
    random_state=42,
    stratify=y_train_42
)

X_fit_raw_46 = X_train_42.iloc[fit_idx_46].reset_index(drop=True)
X_val_raw_46 = X_train_42.iloc[val_idx_46].reset_index(drop=True)

y_fit_46 = y_train_42.iloc[fit_idx_46].reset_index(drop=True)
y_val_46 = y_train_42.iloc[val_idx_46].reset_index(drop=True)

variants_fit_46 = (
    variants_train_42.iloc[fit_idx_46].reset_index(drop=True)
)

variants_val_46 = (
    variants_train_42.iloc[val_idx_46].reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Audit class distribution
# ------------------------------------------------------------

print("\n1. DATASET DIMENSIONS")

for name, features, labels in [
    ("Model-fitting", X_fit_raw_46, y_fit_46),
    ("Validation", X_val_raw_46, y_val_46),
    ("Reserved test", X_test_42, y_test_42)
]:
    benign = int((labels == 0).sum())
    pathogenic = int((labels == 1).sum())

    print(f"\n{name}")
    print(f"Variants: {len(features):,}")
    print(f"Benign: {benign:,}")
    print(f"Pathogenic: {pathogenic:,}")
    print(f"Pathogenic proportion: {pathogenic / len(labels):.4f}")

# ------------------------------------------------------------
# 4. Audit genomic variant overlap
# ------------------------------------------------------------

def variant_key_set_46(frame):
    return set(map(tuple, frame.to_numpy()))

fit_keys_46 = variant_key_set_46(variants_fit_46)
val_keys_46 = variant_key_set_46(variants_val_46)
test_keys_46 = variant_key_set_46(variants_test_42)

fit_val_overlap_46 = len(fit_keys_46 & val_keys_46)
fit_test_overlap_46 = len(fit_keys_46 & test_keys_46)
val_test_overlap_46 = len(val_keys_46 & test_keys_46)

print("\n2. GENOMIC VARIANT OVERLAP")

print("Fit vs validation:", fit_val_overlap_46)
print("Fit vs test:", fit_test_overlap_46)
print("Validation vs test:", val_test_overlap_46)

assert fit_val_overlap_46 == 0
assert fit_test_overlap_46 == 0
assert val_test_overlap_46 == 0

# ------------------------------------------------------------
# 5. Confirm feature integrity
# ------------------------------------------------------------

assert X_fit_raw_46.columns.equals(X_val_raw_46.columns)
assert X_fit_raw_46.columns.equals(X_test_42.columns)

assert len(X_fit_raw_46) + len(X_val_raw_46) == len(X_train_42)

print("\n3. PREPROCESSING SAFEGUARD")
print("Validation data has not been imputed or scaled.")
print("Imputation and scaling must be refitted on fit data only.")
print("Original held-out test set remains unchanged.")

print("\n=== Cell 46 Complete ===")

## Analysis Cell 47

In [ ]:
# ============================================================
# Cell 47 — Leakage-Free Validation Preprocessing
# ============================================================

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

print("=== Cell 47: Leakage-Free Validation Preprocessing ===")

# ------------------------------------------------------------
# 1. Verify inputs
# ------------------------------------------------------------

assert X_fit_raw_46.shape == (128784, 13)
assert X_val_raw_46.shape == (32196, 13)

assert X_fit_raw_46.columns.equals(X_val_raw_46.columns)

assert len(y_fit_46) == len(X_fit_raw_46)
assert len(y_val_46) == len(X_val_raw_46)

feature_names_47 = X_fit_raw_46.columns.tolist()

# ------------------------------------------------------------
# 2. Fit median imputer on model-fitting data ONLY
# ------------------------------------------------------------

imputer_47 = SimpleImputer(strategy="median")

X_fit_imputed_array_47 = imputer_47.fit_transform(
    X_fit_raw_46
)

X_val_imputed_array_47 = imputer_47.transform(
    X_val_raw_46
)

X_fit_imputed_47 = pd.DataFrame(
    X_fit_imputed_array_47,
    columns=feature_names_47
)

X_val_imputed_47 = pd.DataFrame(
    X_val_imputed_array_47,
    columns=feature_names_47
)

# ------------------------------------------------------------
# 3. Fit scaler on imputed model-fitting data ONLY
# ------------------------------------------------------------

scaler_47 = StandardScaler()

X_fit_scaled_array_47 = scaler_47.fit_transform(
    X_fit_imputed_47
)

X_val_scaled_array_47 = scaler_47.transform(
    X_val_imputed_47
)

X_fit_scaled_47 = pd.DataFrame(
    X_fit_scaled_array_47,
    columns=feature_names_47
)

X_val_scaled_47 = pd.DataFrame(
    X_val_scaled_array_47,
    columns=feature_names_47
)

# ------------------------------------------------------------
# 4. Verify shapes and numerical integrity
# ------------------------------------------------------------

print("\n1. DATASET DIMENSIONS")

for name, features in [
    ("Fit — imputed", X_fit_imputed_47),
    ("Validation — imputed", X_val_imputed_47),
    ("Fit — scaled", X_fit_scaled_47),
    ("Validation — scaled", X_val_scaled_47)
]:
    print(f"{name}: {features.shape}")

print("\n2. MISSING AND NONFINITE VALUES")

for name, features in [
    ("Fit — imputed", X_fit_imputed_47),
    ("Validation — imputed", X_val_imputed_47),
    ("Fit — scaled", X_fit_scaled_47),
    ("Validation — scaled", X_val_scaled_47)
]:
    array = features.to_numpy()

    missing = int(features.isna().sum().sum())
    nonfinite = int((~np.isfinite(array)).sum())

    print(
        f"{name}: missing={missing}, "
        f"nonfinite={nonfinite}"
    )

    assert missing == 0
    assert nonfinite == 0

# ------------------------------------------------------------
# 5. Verify standardized fitting data
# ------------------------------------------------------------

fit_means_47 = X_fit_scaled_47.mean().to_numpy()
fit_stds_47 = X_fit_scaled_47.std(ddof=0).to_numpy()

assert np.allclose(fit_means_47, 0, atol=1e-8)
assert np.allclose(fit_stds_47, 1, atol=1e-8)

print("\n3. TRAINING-DERIVED MEDIANS")

median_table_47 = pd.DataFrame({
    "Predictor": feature_names_47,
    "Fit-subset median": imputer_47.statistics_
})

print(
    median_table_47.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n4. LEAKAGE SAFEGUARDS")
print("Median imputer fitted only on model-fitting data.")
print("StandardScaler fitted only on model-fitting data.")
print("Validation data used only for transformation.")
print("Reserved test data not accessed in this cell.")

print("\n=== Cell 47 Complete ===")

## Analysis Cell 48

In [ ]:
# ============================================================
# Cell 48 — Logistic Regression Validation Baseline
# ============================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== Cell 48: Logistic Regression Validation Baseline ===")

# ------------------------------------------------------------
# 1. Verify model-fitting and validation datasets
# ------------------------------------------------------------

assert X_fit_scaled_47.shape == (128784, 13)
assert X_val_scaled_47.shape == (32196, 13)

assert len(y_fit_46) == len(X_fit_scaled_47)
assert len(y_val_46) == len(X_val_scaled_47)

assert np.isfinite(X_fit_scaled_47.to_numpy()).all()
assert np.isfinite(X_val_scaled_47.to_numpy()).all()

# ------------------------------------------------------------
# 2. Train logistic regression on fitting subset ONLY
# ------------------------------------------------------------

logreg_48 = LogisticRegression(
    solver="lbfgs",
    C=1.0,
    max_iter=2000,
    random_state=42
)

logreg_48.fit(X_fit_scaled_47, y_fit_46)

# ------------------------------------------------------------
# 3. Predict validation labels and probabilities
# ------------------------------------------------------------

y_val_pred_48 = logreg_48.predict(X_val_scaled_47)

y_val_prob_48 = logreg_48.predict_proba(
    X_val_scaled_47
)[:, 1]

# ------------------------------------------------------------
# 4. Calculate validation metrics
# ------------------------------------------------------------

results_48 = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_val_46, y_val_pred_48),
    "Balanced accuracy": balanced_accuracy_score(
        y_val_46, y_val_pred_48
    ),
    "Precision": precision_score(
        y_val_46, y_val_pred_48,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val_46, y_val_pred_48,
        zero_division=0
    ),
    "Specificity": recall_score(
        y_val_46, y_val_pred_48,
        pos_label=0,
        zero_division=0
    ),
    "F1": f1_score(
        y_val_46, y_val_pred_48,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val_46, y_val_prob_48
    ),
    "PR-AUC": average_precision_score(
        y_val_46, y_val_prob_48
    )
}

validation_results_48 = pd.DataFrame([results_48])

print("\n1. VALIDATION METRICS")

for metric, value in results_48.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

# ------------------------------------------------------------
# 5. Confusion matrix
# ------------------------------------------------------------

print("\n2. VALIDATION CONFUSION MATRIX")

cm_48 = confusion_matrix(
    y_val_46,
    y_val_pred_48,
    labels=[0, 1]
)

print(
    pd.DataFrame(
        cm_48,
        index=["Actual benign", "Actual pathogenic"],
        columns=["Predicted benign", "Predicted pathogenic"]
    ).to_string()
)

# ------------------------------------------------------------
# 6. Convergence and integrity checks
# ------------------------------------------------------------

print("\n3. MODEL CONVERGENCE")

print("Iterations used:", int(logreg_48.n_iter_[0]))
print("Maximum iterations:", logreg_48.max_iter)

assert logreg_48.n_iter_[0] < logreg_48.max_iter

print("\n4. REPRODUCIBILITY")

print("Model-fitting variants:", len(y_fit_46))
print("Validation variants:", len(y_val_46))
print("Predictors:", X_fit_scaled_47.shape[1])
print("Classification threshold: 0.50")
print("Reserved test set not used.")

print("\n=== Cell 48 Complete ===")

## Analysis Cell 49

In [ ]:
# ============================================================
# Cell 49 — Random Forest Validation Baseline
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== Cell 49: Random Forest Validation Baseline ===")

# ------------------------------------------------------------
# 1. Verify model-fitting and validation data
# ------------------------------------------------------------

assert X_fit_imputed_47.shape == (128784, 13)
assert X_val_imputed_47.shape == (32196, 13)

assert len(y_fit_46) == len(X_fit_imputed_47)
assert len(y_val_46) == len(X_val_imputed_47)

assert X_fit_imputed_47.columns.equals(
    X_val_imputed_47.columns
)

assert np.isfinite(X_fit_imputed_47.to_numpy()).all()
assert np.isfinite(X_val_imputed_47.to_numpy()).all()

# ------------------------------------------------------------
# 2. Train random forest on fitting subset ONLY
# ------------------------------------------------------------

rf_49 = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

start_time_49 = time.perf_counter()

rf_49.fit(X_fit_imputed_47, y_fit_46)

training_seconds_49 = (
    time.perf_counter() - start_time_49
)

# ------------------------------------------------------------
# 3. Predict validation labels and probabilities
# ------------------------------------------------------------

y_val_pred_49 = rf_49.predict(X_val_imputed_47)

y_val_prob_49 = rf_49.predict_proba(
    X_val_imputed_47
)[:, 1]

# ------------------------------------------------------------
# 4. Calculate validation metrics
# ------------------------------------------------------------

results_49 = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_val_46, y_val_pred_49),
    "Balanced accuracy": balanced_accuracy_score(
        y_val_46, y_val_pred_49
    ),
    "Precision": precision_score(
        y_val_46, y_val_pred_49,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val_46, y_val_pred_49,
        zero_division=0
    ),
    "Specificity": recall_score(
        y_val_46, y_val_pred_49,
        pos_label=0,
        zero_division=0
    ),
    "F1": f1_score(
        y_val_46, y_val_pred_49,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val_46, y_val_prob_49
    ),
    "PR-AUC": average_precision_score(
        y_val_46, y_val_prob_49
    )
}

print("\n1. VALIDATION METRICS")

for metric, value in results_49.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

# ------------------------------------------------------------
# 5. Confusion matrix
# ------------------------------------------------------------

print("\n2. VALIDATION CONFUSION MATRIX")

cm_49 = confusion_matrix(
    y_val_46,
    y_val_pred_49,
    labels=[0, 1]
)

print(
    pd.DataFrame(
        cm_49,
        index=["Actual benign", "Actual pathogenic"],
        columns=["Predicted benign", "Predicted pathogenic"]
    ).to_string()
)

# ------------------------------------------------------------
# 6. Feature importance
# ------------------------------------------------------------

print("\n3. RANDOM FOREST FEATURE IMPORTANCE")

importance_49 = pd.DataFrame({
    "Predictor": X_fit_imputed_47.columns,
    "Importance": rf_49.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

print(
    importance_49.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 7. Compare with logistic regression
# ------------------------------------------------------------

validation_results_49 = pd.concat(
    [
        validation_results_48,
        pd.DataFrame([results_49])
    ],
    ignore_index=True
)

print("\n4. MODEL COMPARISON")

print(
    validation_results_49[
        ["Model", "Accuracy", "F1", "ROC-AUC", "PR-AUC"]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 8. Reproducibility
# ------------------------------------------------------------

print("\n5. REPRODUCIBILITY")

print("Model-fitting variants:", len(y_fit_46))
print("Validation variants:", len(y_val_46))
print("Trees:", rf_49.n_estimators)
print("Minimum samples per leaf:", rf_49.min_samples_leaf)
print("Random state:", rf_49.random_state)
print(f"Training time: {training_seconds_49:.1f} seconds")
print("Reserved test set not used.")

print("\n=== Cell 49 Complete ===")

## Analysis Cell 50

In [ ]:
# ============================================================
# Cell 50 — XGBoost Validation Baseline
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== Cell 50: XGBoost Validation Baseline ===")

# ------------------------------------------------------------
# 1. Check XGBoost availability
# ------------------------------------------------------------

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
except ImportError:
    print(
        "XGBoost is not installed in this environment.\n"
        "Install it using: conda install -c conda-forge xgboost\n"
        "Then rerun Cell 50."
    )
else:

    print("\nXGBoost version:", xgb.__version__)

    # --------------------------------------------------------
    # 2. Verify datasets
    # --------------------------------------------------------

    assert X_fit_imputed_47.shape == (128784, 13)
    assert X_val_imputed_47.shape == (32196, 13)

    assert len(y_fit_46) == len(X_fit_imputed_47)
    assert len(y_val_46) == len(X_val_imputed_47)

    assert X_fit_imputed_47.columns.equals(
        X_val_imputed_47.columns
    )

    assert np.isfinite(X_fit_imputed_47.to_numpy()).all()
    assert np.isfinite(X_val_imputed_47.to_numpy()).all()

    # --------------------------------------------------------
    # 3. Train XGBoost on model-fitting data only
    # --------------------------------------------------------

    xgb_50 = XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    start_time_50 = time.perf_counter()

    xgb_50.fit(
        X_fit_imputed_47,
        y_fit_46
    )

    training_seconds_50 = (
        time.perf_counter() - start_time_50
    )

    # --------------------------------------------------------
    # 4. Predict on validation data
    # --------------------------------------------------------

    y_val_pred_50 = xgb_50.predict(
        X_val_imputed_47
    )

    y_val_prob_50 = xgb_50.predict_proba(
        X_val_imputed_47
    )[:, 1]

    # --------------------------------------------------------
    # 5. Calculate validation metrics
    # --------------------------------------------------------

    results_50 = {
        "Model": "XGBoost",
        "Accuracy": accuracy_score(
            y_val_46, y_val_pred_50
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_val_46, y_val_pred_50
        ),
        "Precision": precision_score(
            y_val_46, y_val_pred_50,
            zero_division=0
        ),
        "Recall": recall_score(
            y_val_46, y_val_pred_50,
            zero_division=0
        ),
        "Specificity": recall_score(
            y_val_46, y_val_pred_50,
            pos_label=0,
            zero_division=0
        ),
        "F1": f1_score(
            y_val_46, y_val_pred_50,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_val_46, y_val_prob_50
        ),
        "PR-AUC": average_precision_score(
            y_val_46, y_val_prob_50
        )
    }

    print("\n1. VALIDATION METRICS")

    for metric, value in results_50.items():
        if metric == "Model":
            print(f"{metric}: {value}")
        else:
            print(f"{metric}: {value:.4f}")

    # --------------------------------------------------------
    # 6. Confusion matrix
    # --------------------------------------------------------

    print("\n2. VALIDATION CONFUSION MATRIX")

    cm_50 = confusion_matrix(
        y_val_46,
        y_val_pred_50,
        labels=[0, 1]
    )

    print(
        pd.DataFrame(
            cm_50,
            index=["Actual benign", "Actual pathogenic"],
            columns=["Predicted benign", "Predicted pathogenic"]
        ).to_string()
    )

    # --------------------------------------------------------
    # 7. Feature importance
    # --------------------------------------------------------

    print("\n3. XGBOOST FEATURE IMPORTANCE")

    importance_50 = pd.DataFrame({
        "Predictor": X_fit_imputed_47.columns,
        "Importance": xgb_50.feature_importances_
    }).sort_values(
        "Importance",
        ascending=False
    )

    print(
        importance_50.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

    # --------------------------------------------------------
    # 8. Compare all three models
    # --------------------------------------------------------

    validation_results_50 = pd.concat(
        [
            validation_results_49,
            pd.DataFrame([results_50])
        ],
        ignore_index=True
    )

    print("\n4. MODEL COMPARISON")

    print(
        validation_results_50[
            ["Model", "Accuracy", "F1", "ROC-AUC", "PR-AUC"]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

    # --------------------------------------------------------
    # 9. Reproducibility
    # --------------------------------------------------------

    print("\n5. REPRODUCIBILITY")
    print("Model-fitting variants:", len(y_fit_46))
    print("Validation variants:", len(y_val_46))
    print("Trees:", xgb_50.n_estimators)
    print("Maximum depth:", xgb_50.max_depth)
    print("Learning rate:", xgb_50.learning_rate)
    print("Random state:", xgb_50.random_state)
    print(f"Training time: {training_seconds_50:.1f} seconds")
    print("Reserved test set not used.")

    print("\n=== Cell 50 Complete ===")

## Analysis Cell 51

In [ ]:
# ============================================================
# Cell 51 — LightGBM Validation Baseline
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== Cell 51: LightGBM Validation Baseline ===")

# ------------------------------------------------------------
# 1. Check LightGBM availability
# ------------------------------------------------------------

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
except ImportError:
    print(
        "LightGBM is not installed in this environment.\n"
        "Install it using: conda install -c conda-forge lightgbm\n"
        "Then rerun Cell 51."
    )
else:

    print("\nLightGBM version:", lgb.__version__)

    # --------------------------------------------------------
    # 2. Verify datasets
    # --------------------------------------------------------

    assert X_fit_imputed_47.shape == (128784, 13)
    assert X_val_imputed_47.shape == (32196, 13)

    assert len(y_fit_46) == len(X_fit_imputed_47)
    assert len(y_val_46) == len(X_val_imputed_47)

    assert X_fit_imputed_47.columns.equals(
        X_val_imputed_47.columns
    )

    assert np.isfinite(X_fit_imputed_47.to_numpy()).all()
    assert np.isfinite(X_val_imputed_47.to_numpy()).all()

    # --------------------------------------------------------
    # 3. Train LightGBM on model-fitting data ONLY
    # --------------------------------------------------------

    lgbm_51 = LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        objective="binary",
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    start_time_51 = time.perf_counter()

    lgbm_51.fit(
        X_fit_imputed_47,
        y_fit_46
    )

    training_seconds_51 = (
        time.perf_counter() - start_time_51
    )

    # --------------------------------------------------------
    # 4. Predict on validation data
    # --------------------------------------------------------

    y_val_pred_51 = lgbm_51.predict(
        X_val_imputed_47
    )

    y_val_prob_51 = lgbm_51.predict_proba(
        X_val_imputed_47
    )[:, 1]

    # --------------------------------------------------------
    # 5. Calculate validation metrics
    # --------------------------------------------------------

    results_51 = {
        "Model": "LightGBM",
        "Accuracy": accuracy_score(
            y_val_46, y_val_pred_51
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_val_46, y_val_pred_51
        ),
        "Precision": precision_score(
            y_val_46, y_val_pred_51,
            zero_division=0
        ),
        "Recall": recall_score(
            y_val_46, y_val_pred_51,
            zero_division=0
        ),
        "Specificity": recall_score(
            y_val_46, y_val_pred_51,
            pos_label=0,
            zero_division=0
        ),
        "F1": f1_score(
            y_val_46, y_val_pred_51,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_val_46, y_val_prob_51
        ),
        "PR-AUC": average_precision_score(
            y_val_46, y_val_prob_51
        )
    }

    print("\n1. VALIDATION METRICS")

    for metric, value in results_51.items():
        if metric == "Model":
            print(f"{metric}: {value}")
        else:
            print(f"{metric}: {value:.4f}")

    # --------------------------------------------------------
    # 6. Confusion matrix
    # --------------------------------------------------------

    print("\n2. VALIDATION CONFUSION MATRIX")

    cm_51 = confusion_matrix(
        y_val_46,
        y_val_pred_51,
        labels=[0, 1]
    )

    print(
        pd.DataFrame(
            cm_51,
            index=["Actual benign", "Actual pathogenic"],
            columns=["Predicted benign", "Predicted pathogenic"]
        ).to_string()
    )

    # --------------------------------------------------------
    # 7. Feature importance
    # --------------------------------------------------------

    print("\n3. LIGHTGBM FEATURE IMPORTANCE")

    importance_51 = pd.DataFrame({
        "Predictor": X_fit_imputed_47.columns,
        "Importance": lgbm_51.feature_importances_
    }).sort_values(
        "Importance",
        ascending=False
    )

    print(
        importance_51.to_string(
            index=False
        )
    )

    # --------------------------------------------------------
    # 8. Compare all four models
    # --------------------------------------------------------

    validation_results_51 = pd.concat(
        [
            validation_results_50,
            pd.DataFrame([results_51])
        ],
        ignore_index=True
    )

    print("\n4. MODEL COMPARISON")

    print(
        validation_results_51[
            ["Model", "Accuracy", "F1", "ROC-AUC", "PR-AUC"]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

    # --------------------------------------------------------
    # 9. Reproducibility
    # --------------------------------------------------------

    print("\n5. REPRODUCIBILITY")
    print("Model-fitting variants:", len(y_fit_46))
    print("Validation variants:", len(y_val_46))
    print("Trees:", lgbm_51.n_estimators)
    print("Learning rate:", lgbm_51.learning_rate)
    print("Number of leaves:", lgbm_51.num_leaves)
    print("Random state:", lgbm_51.random_state)
    print(f"Training time: {training_seconds_51:.1f} seconds")
    print("Reserved test set not used.")

    print("\n=== Cell 51 Complete ===")

## Analysis Cell 52

In [ ]:
# ============================================================
# Cell 52 — Linear SVM Validation Baseline
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== Cell 52: Linear SVM Validation Baseline ===")

# ------------------------------------------------------------
# 1. Verify standardized datasets
# ------------------------------------------------------------

assert X_fit_scaled_47.shape == (128784, 13)
assert X_val_scaled_47.shape == (32196, 13)

assert len(y_fit_46) == len(X_fit_scaled_47)
assert len(y_val_46) == len(X_val_scaled_47)

assert X_fit_scaled_47.columns.equals(
    X_val_scaled_47.columns
)

assert np.isfinite(X_fit_scaled_47.to_numpy()).all()
assert np.isfinite(X_val_scaled_47.to_numpy()).all()

# ------------------------------------------------------------
# 2. Configure linear SVM with probability calibration
# ------------------------------------------------------------

svm_base_52 = LinearSVC(
    C=1.0,
    dual=False,
    max_iter=5000,
    random_state=42
)

svm_52 = CalibratedClassifierCV(
    estimator=svm_base_52,
    method="sigmoid",
    cv=3,
    n_jobs=-1
)

# ------------------------------------------------------------
# 3. Fit using model-fitting data ONLY
# ------------------------------------------------------------

start_time_52 = time.perf_counter()

svm_52.fit(
    X_fit_scaled_47,
    y_fit_46
)

training_seconds_52 = (
    time.perf_counter() - start_time_52
)

# ------------------------------------------------------------
# 4. Validation predictions
# ------------------------------------------------------------

y_val_prob_52 = svm_52.predict_proba(
    X_val_scaled_47
)[:, 1]

y_val_pred_52 = (
    y_val_prob_52 >= 0.50
).astype(int)

# ------------------------------------------------------------
# 5. Calculate validation metrics
# ------------------------------------------------------------

results_52 = {
    "Model": "Linear SVM (calibrated)",
    "Accuracy": accuracy_score(
        y_val_46, y_val_pred_52
    ),
    "Balanced accuracy": balanced_accuracy_score(
        y_val_46, y_val_pred_52
    ),
    "Precision": precision_score(
        y_val_46, y_val_pred_52,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val_46, y_val_pred_52,
        zero_division=0
    ),
    "Specificity": recall_score(
        y_val_46, y_val_pred_52,
        pos_label=0,
        zero_division=0
    ),
    "F1": f1_score(
        y_val_46, y_val_pred_52,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val_46, y_val_prob_52
    ),
    "PR-AUC": average_precision_score(
        y_val_46, y_val_prob_52
    )
}

print("\n1. VALIDATION METRICS")

for metric, value in results_52.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

# ------------------------------------------------------------
# 6. Confusion matrix
# ------------------------------------------------------------

print("\n2. VALIDATION CONFUSION MATRIX")

cm_52 = confusion_matrix(
    y_val_46,
    y_val_pred_52,
    labels=[0, 1]
)

print(
    pd.DataFrame(
        cm_52,
        index=["Actual benign", "Actual pathogenic"],
        columns=["Predicted benign", "Predicted pathogenic"]
    ).to_string()
)

# ------------------------------------------------------------
# 7. Update comparison table
# ------------------------------------------------------------

validation_results_52 = pd.concat(
    [
        validation_results_51,
        pd.DataFrame([results_52])
    ],
    ignore_index=True
)

print("\n3. MODEL COMPARISON")

print(
    validation_results_52[
        ["Model", "Accuracy", "F1", "ROC-AUC", "PR-AUC"]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 8. Reproducibility
# ------------------------------------------------------------

print("\n4. REPRODUCIBILITY")
print("Model-fitting variants:", len(y_fit_46))
print("Validation variants:", len(y_val_46))
print("SVM kernel: linear")
print("Regularization C:", svm_base_52.C)
print("Probability calibration: sigmoid")
print("Calibration folds:", svm_52.cv)
print("Random state:", svm_base_52.random_state)
print(f"Training time: {training_seconds_52:.1f} seconds")
print("Reserved test set not used.")

print("\n=== Cell 52 Complete ===")

## Analysis Cell 53

In [ ]:
# ============================================================
# Cell 53 — KNN Validation Baseline
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== Cell 53: KNN Validation Baseline ===")

# ------------------------------------------------------------
# 1. Verify standardized datasets
# ------------------------------------------------------------

assert X_fit_scaled_47.shape == (128784, 13)
assert X_val_scaled_47.shape == (32196, 13)

assert len(y_fit_46) == len(X_fit_scaled_47)
assert len(y_val_46) == len(X_val_scaled_47)

assert X_fit_scaled_47.columns.equals(
    X_val_scaled_47.columns
)

assert np.isfinite(X_fit_scaled_47.to_numpy()).all()
assert np.isfinite(X_val_scaled_47.to_numpy()).all()

# ------------------------------------------------------------
# 2. Configure and fit KNN
# ------------------------------------------------------------

knn_53 = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="euclidean",
    algorithm="brute",
    n_jobs=1
)

start_time_53 = time.perf_counter()

knn_53.fit(
    X_fit_scaled_47,
    y_fit_46
)

training_seconds_53 = (
    time.perf_counter() - start_time_53
)

# ------------------------------------------------------------
# 3. Batched validation probability predictions
# ------------------------------------------------------------

prediction_start_53 = time.perf_counter()

batch_size_53 = 1000
probability_batches_53 = []

for start_53 in range(0, len(X_val_scaled_47), batch_size_53):

    end_53 = min(
        start_53 + batch_size_53,
        len(X_val_scaled_47)
    )

    batch_53 = X_val_scaled_47.iloc[start_53:end_53]

    batch_prob_53 = knn_53.predict_proba(batch_53)[:, 1]

    probability_batches_53.append(batch_prob_53)

y_val_prob_53 = np.concatenate(
    probability_batches_53
)

y_val_pred_53 = (
    y_val_prob_53 >= 0.50
).astype(int)

prediction_seconds_53 = (
    time.perf_counter() - prediction_start_53
)

assert len(y_val_prob_53) == len(y_val_46)
assert np.isfinite(y_val_prob_53).all()

# ------------------------------------------------------------
# 4. Calculate validation metrics
# ------------------------------------------------------------

results_53 = {
    "Model": "KNN (k=5)",
    "Accuracy": accuracy_score(
        y_val_46, y_val_pred_53
    ),
    "Balanced accuracy": balanced_accuracy_score(
        y_val_46, y_val_pred_53
    ),
    "Precision": precision_score(
        y_val_46, y_val_pred_53,
        zero_division=0
    ),
    "Recall": recall_score(
        y_val_46, y_val_pred_53,
        zero_division=0
    ),
    "Specificity": recall_score(
        y_val_46, y_val_pred_53,
        pos_label=0,
        zero_division=0
    ),
    "F1": f1_score(
        y_val_46, y_val_pred_53,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_val_46, y_val_prob_53
    ),
    "PR-AUC": average_precision_score(
        y_val_46, y_val_prob_53
    )
}

print("\n1. VALIDATION METRICS")

for metric, value in results_53.items():
    if metric == "Model":
        print(f"{metric}: {value}")
    else:
        print(f"{metric}: {value:.4f}")

# ------------------------------------------------------------
# 5. Confusion matrix
# ------------------------------------------------------------

print("\n2. VALIDATION CONFUSION MATRIX")

cm_53 = confusion_matrix(
    y_val_46,
    y_val_pred_53,
    labels=[0, 1]
)

print(
    pd.DataFrame(
        cm_53,
        index=["Actual benign", "Actual pathogenic"],
        columns=["Predicted benign", "Predicted pathogenic"]
    ).to_string()
)

# ------------------------------------------------------------
# 6. Compare all six algorithms
# ------------------------------------------------------------

validation_results_53 = pd.concat(
    [
        validation_results_52,
        pd.DataFrame([results_53])
    ],
    ignore_index=True
)

print("\n3. SIX-MODEL VALIDATION COMPARISON")

print(
    validation_results_53[
        [
            "Model",
            "Accuracy",
            "Balanced accuracy",
            "F1",
            "ROC-AUC",
            "PR-AUC"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 7. Reproducibility
# ------------------------------------------------------------

print("\n4. REPRODUCIBILITY")

print("Model-fitting variants:", len(y_fit_46))
print("Validation variants:", len(y_val_46))
print("Number of neighbors:", knn_53.n_neighbors)
print("Distance metric:", knn_53.metric)
print("Weighting:", knn_53.weights)
print("Prediction batch size:", batch_size_53)
print(f"Training time: {training_seconds_53:.1f} seconds")
print(f"Prediction time: {prediction_seconds_53:.1f} seconds")
print("Reserved test set not used.")

print("\n=== Cell 53 Complete ===")

## Analysis Cell 54

In [ ]:
# ============================================================
# Cell 54 — Paired Bootstrap Comparison of Leading Models
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, roc_auc_score

print("=== Cell 54: Paired Bootstrap Model Comparison ===")

# ------------------------------------------------------------
# 1. Collect validation predictions
# ------------------------------------------------------------

y_true_54 = np.asarray(y_val_46, dtype=int)

predictions_54 = {
    "Random Forest": {
        "labels": np.asarray(y_val_pred_49, dtype=int),
        "probabilities": np.asarray(y_val_prob_49, dtype=float)
    },
    "XGBoost": {
        "labels": np.asarray(y_val_pred_50, dtype=int),
        "probabilities": np.asarray(y_val_prob_50, dtype=float)
    },
    "LightGBM": {
        "labels": np.asarray(y_val_pred_51, dtype=int),
        "probabilities": np.asarray(y_val_prob_51, dtype=float)
    }
}

n_54 = len(y_true_54)

assert n_54 == 32196

for model_name, outputs in predictions_54.items():
    assert len(outputs["labels"]) == n_54
    assert len(outputs["probabilities"]) == n_54
    assert np.isfinite(outputs["probabilities"]).all()

# ------------------------------------------------------------
# 2. Calculate observed differences
# ------------------------------------------------------------

comparisons_54 = [
    ("Random Forest", "XGBoost"),
    ("Random Forest", "LightGBM"),
    ("LightGBM", "XGBoost")
]

observed_54 = {}

for model_a, model_b in comparisons_54:

    f1_a = f1_score(
        y_true_54,
        predictions_54[model_a]["labels"]
    )

    f1_b = f1_score(
        y_true_54,
        predictions_54[model_b]["labels"]
    )

    auc_a = roc_auc_score(
        y_true_54,
        predictions_54[model_a]["probabilities"]
    )

    auc_b = roc_auc_score(
        y_true_54,
        predictions_54[model_b]["probabilities"]
    )

    observed_54[(model_a, model_b)] = {
        "F1": f1_a - f1_b,
        "ROC-AUC": auc_a - auc_b
    }

# ------------------------------------------------------------
# 3. Paired bootstrap
# ------------------------------------------------------------

n_bootstrap_54 = 1000
rng_54 = np.random.default_rng(42)

bootstrap_54 = {
    (model_a, model_b, metric): []
    for model_a, model_b in comparisons_54
    for metric in ["F1", "ROC-AUC"]
}

for iteration_54 in range(n_bootstrap_54):

    sample_idx_54 = rng_54.integers(
        0, n_54, size=n_54
    )

    y_sample_54 = y_true_54[sample_idx_54]

    # Both classes are needed to calculate ROC-AUC.
    if np.unique(y_sample_54).size < 2:
        continue

    sample_metrics_54 = {}

    for model_name, outputs in predictions_54.items():

        sampled_labels = outputs["labels"][sample_idx_54]
        sampled_probs = outputs["probabilities"][sample_idx_54]

        sample_metrics_54[model_name] = {
            "F1": f1_score(
                y_sample_54,
                sampled_labels,
                zero_division=0
            ),
            "ROC-AUC": roc_auc_score(
                y_sample_54,
                sampled_probs
            )
        }

    for model_a, model_b in comparisons_54:

        for metric in ["F1", "ROC-AUC"]:

            difference = (
                sample_metrics_54[model_a][metric]
                - sample_metrics_54[model_b][metric]
            )

            bootstrap_54[
                (model_a, model_b, metric)
            ].append(difference)

# ------------------------------------------------------------
# 4. Summarize bootstrap confidence intervals
# ------------------------------------------------------------

rows_54 = []

for model_a, model_b in comparisons_54:

    for metric in ["F1", "ROC-AUC"]:

        differences = np.asarray(
            bootstrap_54[(model_a, model_b, metric)]
        )

        lower_54, upper_54 = np.percentile(
            differences,
            [2.5, 97.5]
        )

        rows_54.append({
            "Comparison": f"{model_a} - {model_b}",
            "Metric": metric,
            "Observed difference": observed_54[
                (model_a, model_b)
            ][metric],
            "95% CI lower": lower_54,
            "95% CI upper": upper_54,
            "CI excludes zero": (
                lower_54 > 0 or upper_54 < 0
            )
        })

bootstrap_results_54 = pd.DataFrame(rows_54)

print("\n1. PAIRED BOOTSTRAP RESULTS")

print(
    bootstrap_results_54.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

# ------------------------------------------------------------
# 5. Reproducibility
# ------------------------------------------------------------

print("\n2. REPRODUCIBILITY")
print("Validation variants:", n_54)
print("Bootstrap resamples:", n_bootstrap_54)
print("Random seed: 42")
print("Sampling: paired, with replacement")
print("Confidence interval: percentile, 95%")
print("Models retrained: No")
print("Reserved test set used: No")

print("\n=== Cell 54 Complete ===")

## Analysis Cell 55

In [ ]:
# ============================================================
# Cell 55 — Probability Calibration Assessment
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import brier_score_loss, log_loss
from sklearn.calibration import calibration_curve

print("=== Cell 55: Probability Calibration Assessment ===")

# ------------------------------------------------------------
# 1. Collect validation probabilities
# ------------------------------------------------------------

y_true_55 = np.asarray(y_val_46, dtype=int)

probabilities_55 = {
    "Random Forest": np.asarray(y_val_prob_49, dtype=float),
    "XGBoost": np.asarray(y_val_prob_50, dtype=float),
    "LightGBM": np.asarray(y_val_prob_51, dtype=float)
}

assert len(y_true_55) == 32196
assert set(np.unique(y_true_55)) == {0, 1}

for name_55, probs_55 in probabilities_55.items():
    assert len(probs_55) == len(y_true_55)
    assert np.isfinite(probs_55).all()
    assert ((probs_55 >= 0) & (probs_55 <= 1)).all()

# ------------------------------------------------------------
# 2. Calculate calibration metrics
# ------------------------------------------------------------

calibration_rows_55 = []

for name_55, probs_55 in probabilities_55.items():

    calibration_rows_55.append({
        "Model": name_55,
        "Brier score": brier_score_loss(
            y_true_55, probs_55
        ),
        "Log loss": log_loss(
            y_true_55, probs_55,
            labels=[0, 1]
        ),
        "Mean predicted probability": probs_55.mean(),
        "Observed pathogenic proportion": y_true_55.mean()
    })

calibration_results_55 = pd.DataFrame(
    calibration_rows_55
).sort_values(
    "Brier score"
).reset_index(drop=True)

print("\n1. CALIBRATION METRICS")
print("Lower Brier score and log loss are better.\n")

print(
    calibration_results_55.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

# ------------------------------------------------------------
# 3. Plot calibration curves
# ------------------------------------------------------------

fig_55, ax_55 = plt.subplots(figsize=(8, 7))

ax_55.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Perfect calibration"
)

for name_55, probs_55 in probabilities_55.items():

    observed_55, predicted_55 = calibration_curve(
        y_true_55,
        probs_55,
        n_bins=10,
        strategy="quantile"
    )

    ax_55.plot(
        predicted_55,
        observed_55,
        marker="o",
        linewidth=1.5,
        label=name_55
    )

ax_55.set_xlabel("Mean predicted pathogenic probability")
ax_55.set_ylabel("Observed pathogenic proportion")
ax_55.set_title(
    "Validation Probability Calibration — PathoPredictor"
)

ax_55.set_xlim(0, 1)
ax_55.set_ylim(0, 1)
ax_55.grid(alpha=0.25)
ax_55.legend()

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. Report calibration interpretation safeguards
# ------------------------------------------------------------

print("\n2. REPRODUCIBILITY")

print("Validation variants:", len(y_true_55))
print("Calibration bins: 10")
print("Binning strategy: quantile")
print("Models retrained: No")
print("Probabilities recalibrated: No")
print("Reserved test set used: No")

print("\n3. INTERPRETATION NOTE")
print(
    "These are internal validation calibration diagnostics. "
    "They do not establish clinical probability calibration."
)

print("\n=== Cell 55 Complete ===")

## Analysis Cell 56

In [ ]:
# ============================================================
# Cell 56 — Record Model Selection and Validation Results
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

print("=== Cell 56: Record Model Selection ===")

# ------------------------------------------------------------
# 1. Verify all six validation results
# ------------------------------------------------------------

assert len(validation_results_53) == 6
assert validation_results_53["Model"].is_unique

required_metrics_56 = [
    "Accuracy",
    "Balanced accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "ROC-AUC",
    "PR-AUC"
]

assert all(
    metric in validation_results_53.columns
    for metric in required_metrics_56
)

assert np.isfinite(
    validation_results_53[required_metrics_56].to_numpy(
        dtype=float
    )
).all()

# ------------------------------------------------------------
# 2. Rank models by primary selection criterion
# ------------------------------------------------------------

primary_metric_56 = "ROC-AUC"

ranked_results_56 = (
    validation_results_53
    .sort_values(
        ["ROC-AUC", "PR-AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

selected_model_56 = ranked_results_56.loc[0, "Model"]

assert selected_model_56 == "LightGBM"

print("\n1. VALIDATION RANKING")

print(
    ranked_results_56[
        ["Model", "Accuracy", "F1", "ROC-AUC", "PR-AUC"]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

# ------------------------------------------------------------
# 3. Create model-selection record
# ------------------------------------------------------------

selection_record_56 = {
    "analysis": "PathoPredictor reconstruction",
    "dataset_status": (
        "Reconstructed dataset; not an exact reproduction "
        "of the published 28-predictor dataset"
    ),
    "number_of_predictors": 13,
    "model_fitting_variants": int(len(y_fit_46)),
    "validation_variants": int(len(y_val_46)),
    "reserved_test_variants": int(len(y_test_42)),
    "candidate_models": ranked_results_56["Model"].tolist(),
    "primary_selection_metric": primary_metric_56,
    "selected_model": selected_model_56,
    "selected_model_validation_roc_auc": float(
        ranked_results_56.loc[0, "ROC-AUC"]
    ),
    "selected_model_validation_pr_auc": float(
        ranked_results_56.loc[0, "PR-AUC"]
    ),
    "selected_model_validation_f1": float(
        ranked_results_56.loc[0, "F1"]
    ),
    "selected_model_validation_brier_score": float(
        calibration_results_55.loc[
            calibration_results_55["Model"] == "LightGBM",
            "Brier score"
        ].iloc[0]
    ),
    "selected_model_validation_log_loss": float(
        calibration_results_55.loc[
            calibration_results_55["Model"] == "LightGBM",
            "Log loss"
        ].iloc[0]
    ),
    "classification_threshold": 0.50,
    "random_state": 42,
    "selection_note": (
        "LightGBM selected provisionally by highest validation "
        "ROC-AUC among six baseline algorithms. "
        "No hyperparameter tuning was performed."
    ),
    "test_set_policy": (
        "No further test-set evaluation until model selection "
        "and analysis plan are finalized. An earlier exploratory "
        "logistic regression test evaluation was performed "
        "in Cell 45 and must be disclosed."
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

# ------------------------------------------------------------
# 4. Save outputs
# ------------------------------------------------------------

output_dir_56 = TABLES_DIR
output_dir_56.mkdir(
    parents=True,
    exist_ok=True
)

comparison_path_56 = (
    output_dir_56 / "validation_model_comparison.csv"
)

selection_path_56 = (
    MODELS_DIR / "model_selection_record.json"
)

ranked_results_56.to_csv(
    comparison_path_56,
    index=False
)

with open(
    selection_path_56,
    "w",
    encoding="utf-8"
) as file_56:
    json.dump(
        selection_record_56,
        file_56,
        indent=2
    )

# ------------------------------------------------------------
# 5. Verify saved files
# ------------------------------------------------------------

assert comparison_path_56.is_file()
assert selection_path_56.is_file()

saved_results_56 = pd.read_csv(
    comparison_path_56
)

assert len(saved_results_56) == 6

with open(
    selection_path_56,
    "r",
    encoding="utf-8"
) as file_56:
    saved_selection_56 = json.load(file_56)

assert saved_selection_56["selected_model"] == "LightGBM"

print("\n2. MODEL SELECTION")
print("Primary metric:", primary_metric_56)
print("Provisionally selected model:", selected_model_56)
print("Hyperparameter tuning performed: No")
print("Classification threshold: 0.50")

print("\n3. SAVED FILES")
print(comparison_path_56)
print(selection_path_56)

print("\n4. REPRODUCIBILITY")
print("Validation variants:", len(y_val_46))
print("Reserved test variants:", len(y_test_42))
print("Test predictions accessed in this cell: No")
print("Earlier exploratory test evaluation: Cell 45")

print("\n=== Cell 56 Complete ===")

## Analysis Cell 57

In [ ]:
# ============================================================
# Cell 57 — Final LightGBM Training
# ============================================================

import time
import numpy as np
import pandas as pd

import lightgbm as lgb

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier

print("=== Cell 57: Final LightGBM Training ===")

# ------------------------------------------------------------
# 1. Verify full development dataset
# ------------------------------------------------------------

assert X_train_42.shape == (160980, 13)
assert len(y_train_42) == 160980

assert X_test_42.shape == (40245, 13)
assert len(y_test_42) == 40245

assert X_train_42.columns.equals(X_test_42.columns)

assert set(np.unique(y_train_42)) == {0, 1}

# Verify the development set consists of fit + validation.
assert (
    len(y_fit_46) + len(y_val_46)
    == len(y_train_42)
)

feature_names_57 = X_train_42.columns.tolist()

# ------------------------------------------------------------
# 2. Construct final preprocessing + model pipeline
# ------------------------------------------------------------

final_lgbm_57 = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "classifier",
            LGBMClassifier(
                n_estimators=200,
                learning_rate=0.05,
                num_leaves=31,
                max_depth=-1,
                min_child_samples=20,
                subsample=0.8,
                subsample_freq=1,
                colsample_bytree=0.8,
                objective="binary",
                random_state=42,
                n_jobs=-1,
                verbosity=-1
            )
        )
    ]
)

# ------------------------------------------------------------
# 3. Fit on all development variants
# ------------------------------------------------------------

start_time_57 = time.perf_counter()

final_lgbm_57.fit(
    X_train_42,
    y_train_42
)

training_seconds_57 = (
    time.perf_counter() - start_time_57
)

# ------------------------------------------------------------
# 4. Verify fitted components
# ------------------------------------------------------------

final_imputer_57 = final_lgbm_57.named_steps["imputer"]
final_classifier_57 = final_lgbm_57.named_steps["classifier"]

assert len(final_imputer_57.statistics_) == 13
assert final_classifier_57.n_features_in_ == 13

assert (
    list(final_lgbm_57.feature_names_in_)
    == feature_names_57
)

# ------------------------------------------------------------
# 5. Report training configuration
# ------------------------------------------------------------

print("\n1. FINAL MODEL CONFIGURATION")

print("Algorithm: LightGBM")
print("Development variants:", len(y_train_42))
print("Predictors:", len(feature_names_57))
print("Trees:", final_classifier_57.n_estimators)
print("Learning rate:", final_classifier_57.learning_rate)
print("Number of leaves:", final_classifier_57.num_leaves)
print("Minimum child samples:", final_classifier_57.min_child_samples)
print("Random state:", final_classifier_57.random_state)
print(f"Training time: {training_seconds_57:.2f} seconds")

print("\n2. DEVELOPMENT CLASS DISTRIBUTION")

print("Benign:", int((y_train_42 == 0).sum()))
print("Pathogenic:", int((y_train_42 == 1).sum()))

# ------------------------------------------------------------
# 6. Report final imputation medians
# ------------------------------------------------------------

print("\n3. FINAL TRAINING-DERIVED MEDIANS")

medians_57 = pd.DataFrame({
    "Predictor": feature_names_57,
    "Median": final_imputer_57.statistics_
})

print(
    medians_57.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ------------------------------------------------------------
# 7. Confirm no test evaluation occurred
# ------------------------------------------------------------

print("\n4. REPRODUCIBILITY")

print("Model-selection source: Cell 56")
print("Model selected: LightGBM")
print("Preprocessing: median imputation")
print("Preprocessing fitted on development data only.")
print("Validation subset included in final training: Yes")
print("Reserved test variants:", len(y_test_42))
print("Test predictions generated: No")
print("Test labels used for fitting: No")

print("\n=== Cell 57 Complete ===")

## Analysis Cell 58

In [ ]:
# ============================================================
# Cell 58 — Save and Verify Final LightGBM Model
# ============================================================

import json
import platform
from pathlib import Path
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb

print("=== Cell 58: Save and Verify Final LightGBM Model ===")

# ------------------------------------------------------------
# 1. Define output files
# ------------------------------------------------------------

output_dir_58 = MODELS_DIR
output_dir_58.mkdir(parents=True, exist_ok=True)

model_path_58 = output_dir_58 / "pathopredictor_lightgbm_pipeline.joblib"
metadata_path_58 = output_dir_58 / "pathopredictor_lightgbm_metadata.json"

# ------------------------------------------------------------
# 2. Verify final model and feature schema
# ------------------------------------------------------------

assert final_lgbm_57 is not None
assert X_train_42.shape == (160980, 13)
assert X_test_42.shape == (40245, 13)

feature_names_58 = X_train_42.columns.tolist()

assert feature_names_58 == feature_names_57
assert list(final_lgbm_57.feature_names_in_) == feature_names_58

assert list(final_lgbm_57.named_steps.keys()) == [
    "imputer",
    "classifier"
]

# ------------------------------------------------------------
# 3. Save fitted pipeline
# ------------------------------------------------------------

joblib.dump(
    final_lgbm_57,
    model_path_58,
    compress=3
)

assert model_path_58.is_file()

# ------------------------------------------------------------
# 4. Create reproducibility metadata
# ------------------------------------------------------------

classifier_58 = final_lgbm_57.named_steps["classifier"]
imputer_58 = final_lgbm_57.named_steps["imputer"]

metadata_58 = {
    "project": "PathoPredictor reconstruction",
    "reconstruction_status": (
        "Reconstructed 13-predictor model; not an exact "
        "reproduction of the published 28-predictor model."
    ),
    "model": "LightGBM",
    "model_selection": {
        "source_cell": 56,
        "criterion": "validation ROC-AUC",
        "validation_roc_auc": float(
            ranked_results_56.loc[0, "ROC-AUC"]
        ),
        "hyperparameter_tuning": False
    },
    "dataset": {
        "development_variants": int(len(y_train_42)),
        "reserved_test_variants": int(len(y_test_42)),
        "number_of_predictors": len(feature_names_58),
        "development_benign": int((y_train_42 == 0).sum()),
        "development_pathogenic": int((y_train_42 == 1).sum())
    },
    "feature_names_in_order": feature_names_58,
    "preprocessing": {
        "missing_value_strategy": "median",
        "fitted_on": "development partition only",
        "scaling": "none",
        "training_medians": {
            name: float(value)
            for name, value in zip(
                feature_names_58,
                imputer_58.statistics_
            )
        }
    },
    "classifier_hyperparameters": {
        "n_estimators": classifier_58.n_estimators,
        "learning_rate": classifier_58.learning_rate,
        "num_leaves": classifier_58.num_leaves,
        "max_depth": classifier_58.max_depth,
        "min_child_samples": classifier_58.min_child_samples,
        "subsample": classifier_58.subsample,
        "subsample_freq": classifier_58.subsample_freq,
        "colsample_bytree": classifier_58.colsample_bytree,
        "objective": classifier_58.objective,
        "random_state": classifier_58.random_state
    },
    "classification_threshold": 0.50,
    "software_versions": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "lightgbm": lgb.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "joblib": joblib.__version__
    },
    "methodological_notes": [
        "Variant-level stratified train/test split.",
        "Model selected using a separate validation subset.",
        "Final pipeline refitted on all development variants.",
        "Test partition excluded from final model fitting.",
        "Earlier exploratory logistic regression test evaluation "
        "was performed in Cell 45.",
        "Constituent pathogenicity scores may have overlapping "
        "training sources with ClinVar labels.",
        "Gene-level independence and external clinical validity "
        "have not been established."
    ],
    "saved_utc": datetime.now(timezone.utc).isoformat()
}

with open(metadata_path_58, "w", encoding="utf-8") as file_58:
    json.dump(metadata_58, file_58, indent=2)

assert metadata_path_58.is_file()

# ------------------------------------------------------------
# 5. Reload pipeline and verify identical predictions
# ------------------------------------------------------------

reloaded_lgbm_58 = joblib.load(model_path_58)

sample_58 = X_train_42.iloc[:100].copy()

original_probabilities_58 = final_lgbm_57.predict_proba(
    sample_58
)[:, 1]

reloaded_probabilities_58 = reloaded_lgbm_58.predict_proba(
    sample_58
)[:, 1]

max_difference_58 = float(
    np.max(
        np.abs(
            original_probabilities_58
            - reloaded_probabilities_58
        )
    )
)

assert np.allclose(
    original_probabilities_58,
    reloaded_probabilities_58,
    rtol=0,
    atol=1e-12
)

# ------------------------------------------------------------
# 6. Verify saved metadata
# ------------------------------------------------------------

with open(metadata_path_58, "r", encoding="utf-8") as file_58:
    loaded_metadata_58 = json.load(file_58)

assert loaded_metadata_58["model"] == "LightGBM"

assert (
    loaded_metadata_58["feature_names_in_order"]
    == feature_names_58
)

# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print("\n1. SAVED ARTIFACTS")
print("Model:", model_path_58)
print("Metadata:", metadata_path_58)

print("\n2. FILE SIZES")
print(
    "Model size:",
    f"{model_path_58.stat().st_size / (1024**2):.2f} MiB"
)
print(
    "Metadata size:",
    f"{metadata_path_58.stat().st_size / 1024:.2f} KiB"
)

print("\n3. RELOAD VERIFICATION")
print("Development samples checked:", len(sample_58))
print(f"Maximum probability difference: {max_difference_58:.12f}")
print("Reloaded predictions match: Yes")

print("\n4. REPRODUCIBILITY")
print("Model: LightGBM")
print("Predictors:", len(feature_names_58))
print("Development variants:", len(y_train_42))
print("Reserved test variants:", len(y_test_42))
print("Test predictions generated: No")
print("Test labels accessed: No")

print("\n=== Cell 58 Complete ===")

## Analysis Cell 59

In [ ]:
# ============================================================
# Cell 59 — Final Held-Out Test Evaluation
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    brier_score_loss,
    log_loss
)

print("=== Cell 59: Final Held-Out Test Evaluation ===")

# ------------------------------------------------------------
# 1. Verify model and reserved test data
# ------------------------------------------------------------

assert X_test_42.shape == (40245, 13)
assert len(y_test_42) == 40245
assert X_test_42.columns.tolist() == feature_names_57

y_true_59 = np.asarray(y_test_42, dtype=int)

assert set(np.unique(y_true_59)) == {0, 1}

# Verify no genomic variant overlaps between partitions.
train_keys_59 = set(
    map(tuple, variants_train_42[
        ["chrom", "pos", "ref", "alt"]
    ].astype(str).to_numpy())
)

test_keys_59 = set(
    map(tuple, variants_test_42[
        ["chrom", "pos", "ref", "alt"]
    ].astype(str).to_numpy())
)

assert train_keys_59.isdisjoint(test_keys_59)

# ------------------------------------------------------------
# 2. Generate final predictions
# ------------------------------------------------------------

threshold_59 = 0.50

y_prob_59 = final_lgbm_57.predict_proba(
    X_test_42
)[:, 1]

y_pred_59 = (
    y_prob_59 >= threshold_59
).astype(int)

assert len(y_prob_59) == len(y_true_59)
assert np.isfinite(y_prob_59).all()
assert ((y_prob_59 >= 0) & (y_prob_59 <= 1)).all()

# ------------------------------------------------------------
# 3. Calculate held-out test metrics
# ------------------------------------------------------------

cm_59 = confusion_matrix(
    y_true_59,
    y_pred_59,
    labels=[0, 1]
)

tn_59, fp_59, fn_59, tp_59 = cm_59.ravel()

results_59 = {
    "Model": "LightGBM (final)",
    "Accuracy": accuracy_score(y_true_59, y_pred_59),
    "Balanced accuracy": balanced_accuracy_score(
        y_true_59, y_pred_59
    ),
    "Precision": precision_score(
        y_true_59, y_pred_59,
        zero_division=0
    ),
    "Recall": recall_score(
        y_true_59, y_pred_59,
        zero_division=0
    ),
    "Specificity": tn_59 / (tn_59 + fp_59),
    "F1": f1_score(
        y_true_59, y_pred_59,
        zero_division=0
    ),
    "ROC-AUC": roc_auc_score(
        y_true_59, y_prob_59
    ),
    "PR-AUC": average_precision_score(
        y_true_59, y_prob_59
    ),
    "Brier score": brier_score_loss(
        y_true_59, y_prob_59
    ),
    "Log loss": log_loss(
        y_true_59, y_prob_59,
        labels=[0, 1]
    )
}

test_results_59 = pd.DataFrame([results_59])

print("\n1. FINAL HELD-OUT TEST METRICS")

for metric_59, value_59 in results_59.items():
    if metric_59 == "Model":
        print(f"{metric_59}: {value_59}")
    else:
        print(f"{metric_59}: {value_59:.5f}")

# ------------------------------------------------------------
# 4. Confusion matrix
# ------------------------------------------------------------

print("\n2. TEST CONFUSION MATRIX")

cm_table_59 = pd.DataFrame(
    cm_59,
    index=["Actual benign", "Actual pathogenic"],
    columns=["Predicted benign", "Predicted pathogenic"]
)

print(cm_table_59.to_string())

# ------------------------------------------------------------
# 5. Test-set class distribution
# ------------------------------------------------------------

print("\n3. TEST CLASS DISTRIBUTION")

print("Total variants:", len(y_true_59))
print("Benign:", int((y_true_59 == 0).sum()))
print("Pathogenic:", int((y_true_59 == 1).sum()))
print(
    "Observed pathogenic proportion:",
    f"{y_true_59.mean():.5f}"
)
print(
    "Mean predicted pathogenic probability:",
    f"{y_prob_59.mean():.5f}"
)

# ------------------------------------------------------------
# 6. Save per-variant test predictions
# ------------------------------------------------------------

output_dir_59 = TABLES_DIR
output_dir_59.mkdir(parents=True, exist_ok=True)

test_metrics_path_59 = (
    output_dir_59 / "final_lightgbm_test_metrics.csv"
)

test_predictions_path_59 = (
    output_dir_59 / "final_lightgbm_test_predictions.csv"
)

test_results_59.to_csv(
    test_metrics_path_59,
    index=False
)

test_predictions_59 = (
    variants_test_42[
        ["chrom", "pos", "ref", "alt"]
    ]
    .reset_index(drop=True)
    .copy()
)

test_predictions_59["true_label"] = y_true_59
test_predictions_59["predicted_label"] = y_pred_59
test_predictions_59["pathogenic_probability"] = y_prob_59

assert len(test_predictions_59) == 40245

test_predictions_59.to_csv(
    test_predictions_path_59,
    index=False
)

# ------------------------------------------------------------
# 7. Save confusion matrix
# ------------------------------------------------------------

confusion_path_59 = (
    output_dir_59 / "final_lightgbm_test_confusion_matrix.csv"
)

cm_table_59.to_csv(confusion_path_59)

assert test_metrics_path_59.is_file()
assert test_predictions_path_59.is_file()
assert confusion_path_59.is_file()

# ------------------------------------------------------------
# 8. Reproducibility report
# ------------------------------------------------------------

print("\n4. SAVED FILES")
print(test_metrics_path_59)
print(test_predictions_path_59)
print(confusion_path_59)

print("\n5. REPRODUCIBILITY")
print("Final model: LightGBM")
print("Model-selection criterion: validation ROC-AUC")
print("Classification threshold:", threshold_59)
print("Development variants:", len(y_train_42))
print("Held-out test variants:", len(y_true_59))
print("Development/test genomic overlap: 0")
print("Model refitted on test data: No")
print("Hyperparameters changed after validation: No")
print("Earlier exploratory test access: Cell 45 (disclose)")

print("\n=== Cell 59 Complete ===")

## Analysis Cell 60

In [ ]:
# ============================================================
# Cell 60 — Final Test Performance Confidence Intervals
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== Cell 60: Final Test Confidence Intervals ===")

# ------------------------------------------------------------
# 1. Verify saved final test predictions
# ------------------------------------------------------------

y_true_60 = np.asarray(y_true_59, dtype=int)
y_pred_60 = np.asarray(y_pred_59, dtype=int)
y_prob_60 = np.asarray(y_prob_59, dtype=float)

assert len(y_true_60) == 40245
assert len(y_pred_60) == len(y_true_60)
assert len(y_prob_60) == len(y_true_60)
assert set(np.unique(y_true_60)) == {0, 1}
assert np.isfinite(y_prob_60).all()

# ------------------------------------------------------------
# 2. Define performance metrics
# ------------------------------------------------------------

def calculate_metrics_60(y_true, y_pred, y_prob):

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Sensitivity": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(
            y_true, y_prob
        )
    }

observed_60 = calculate_metrics_60(
    y_true_60, y_pred_60, y_prob_60
)

# ------------------------------------------------------------
# 3. Stratified bootstrap
# ------------------------------------------------------------

n_bootstrap_60 = 1000
rng_60 = np.random.default_rng(42)

benign_idx_60 = np.flatnonzero(y_true_60 == 0)
pathogenic_idx_60 = np.flatnonzero(y_true_60 == 1)

bootstrap_metrics_60 = {
    metric: []
    for metric in observed_60
}

for iteration_60 in range(n_bootstrap_60):

    sampled_benign_60 = rng_60.choice(
        benign_idx_60,
        size=len(benign_idx_60),
        replace=True
    )

    sampled_pathogenic_60 = rng_60.choice(
        pathogenic_idx_60,
        size=len(pathogenic_idx_60),
        replace=True
    )

    sampled_idx_60 = np.concatenate([
        sampled_benign_60,
        sampled_pathogenic_60
    ])

    sampled_metrics_60 = calculate_metrics_60(
        y_true_60[sampled_idx_60],
        y_pred_60[sampled_idx_60],
        y_prob_60[sampled_idx_60]
    )

    for metric_60, value_60 in sampled_metrics_60.items():
        bootstrap_metrics_60[metric_60].append(value_60)

# ------------------------------------------------------------
# 4. Calculate percentile confidence intervals
# ------------------------------------------------------------

rows_60 = []

for metric_60, observed_value_60 in observed_60.items():

    lower_60, upper_60 = np.percentile(
        bootstrap_metrics_60[metric_60],
        [2.5, 97.5]
    )

    rows_60.append({
        "Metric": metric_60,
        "Observed": observed_value_60,
        "95% CI lower": lower_60,
        "95% CI upper": upper_60
    })

confidence_intervals_60 = pd.DataFrame(rows_60)

print("\n1. FINAL TEST METRICS WITH 95% CONFIDENCE INTERVALS")

print(
    confidence_intervals_60.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

ci_path_60 = (
    output_dir_59
    / "final_lightgbm_test_confidence_intervals.csv"
)

confidence_intervals_60.to_csv(
    ci_path_60,
    index=False
)

assert ci_path_60.is_file()

print("\n2. SAVED FILE")
print(ci_path_60)

# ------------------------------------------------------------
# 6. Reproducibility
# ------------------------------------------------------------

print("\n3. REPRODUCIBILITY")
print("Test variants:", len(y_true_60))
print("Bootstrap resamples:", n_bootstrap_60)
print("Random seed: 42")
print("Bootstrap method: stratified by true class")
print("Confidence interval: percentile, 95%")
print("Model retrained: No")
print("Threshold changed: No")
print("Class proportions fixed across resamples: Yes")

print("\n4. LIMITATION")
print(
    "Intervals assume independent variants within each class. "
    "They do not account for gene-level clustering or "
    "uncertainty arising from dataset reconstruction."
)

print("\n=== Cell 60 Complete ===")

## Analysis Cell 61

In [ ]:
# ============================================================
# Cell 61 — Chromosome-Level Test Performance
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("=== Cell 61: Chromosome-Level Test Performance ===")

# ------------------------------------------------------------
# 1. Prepare chromosome-level test predictions
# ------------------------------------------------------------

chromosome_test_61 = test_predictions_59.copy()

required_columns_61 = [
    "chrom",
    "true_label",
    "predicted_label",
    "pathogenic_probability"
]

assert all(
    column in chromosome_test_61.columns
    for column in required_columns_61
)

assert len(chromosome_test_61) == 40245

chromosome_test_61["chrom"] = (
    chromosome_test_61["chrom"]
    .astype(str)
    .str.replace(r"^chr", "", regex=True)
)

# ------------------------------------------------------------
# 2. Calculate metrics for each chromosome
# ------------------------------------------------------------

rows_61 = []

for chrom_61, group_61 in chromosome_test_61.groupby(
    "chrom",
    sort=False
):

    y_true_chr_61 = group_61["true_label"].to_numpy(
        dtype=int
    )

    y_pred_chr_61 = group_61["predicted_label"].to_numpy(
        dtype=int
    )

    y_prob_chr_61 = group_61[
        "pathogenic_probability"
    ].to_numpy(dtype=float)

    tn_61, fp_61, fn_61, tp_61 = confusion_matrix(
        y_true_chr_61,
        y_pred_chr_61,
        labels=[0, 1]
    ).ravel()

    has_both_classes_61 = (
        len(np.unique(y_true_chr_61)) == 2
    )

    rows_61.append({
        "Chromosome": chrom_61,
        "Variants": len(group_61),
        "Benign": int((y_true_chr_61 == 0).sum()),
        "Pathogenic": int((y_true_chr_61 == 1).sum()),
        "Accuracy": accuracy_score(
            y_true_chr_61,
            y_pred_chr_61
        ),
        "Sensitivity": (
            tp_61 / (tp_61 + fn_61)
            if (tp_61 + fn_61) > 0
            else np.nan
        ),
        "Specificity": (
            tn_61 / (tn_61 + fp_61)
            if (tn_61 + fp_61) > 0
            else np.nan
        ),
        "F1": (
            f1_score(
                y_true_chr_61,
                y_pred_chr_61,
                zero_division=0
            )
            if (tp_61 + fn_61) > 0
            else np.nan
        ),
        "ROC-AUC": (
            roc_auc_score(
                y_true_chr_61,
                y_prob_chr_61
            )
            if has_both_classes_61
            else np.nan
        )
    })

chromosome_results_61 = pd.DataFrame(rows_61)

# ------------------------------------------------------------
# 3. Sort chromosomes in genomic order
# ------------------------------------------------------------

chromosome_order_61 = {
    str(chrom): chrom
    for chrom in range(1, 23)
}

chromosome_order_61.update({
    "X": 23,
    "Y": 24,
    "MT": 25,
    "M": 25
})

chromosome_results_61["_order"] = (
    chromosome_results_61["Chromosome"]
    .map(chromosome_order_61)
    .fillna(99)
)

chromosome_results_61 = (
    chromosome_results_61
    .sort_values(["_order", "Chromosome"])
    .drop(columns="_order")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Verify subgroup totals
# ------------------------------------------------------------

assert chromosome_results_61["Variants"].sum() == 40245

assert (
    chromosome_results_61["Benign"].sum()
    == int((y_true_59 == 0).sum())
)

assert (
    chromosome_results_61["Pathogenic"].sum()
    == int((y_true_59 == 1).sum())
)

# ------------------------------------------------------------
# 5. Display chromosome-level performance
# ------------------------------------------------------------

print("\n1. CHROMOSOME-LEVEL TEST RESULTS")

print(
    chromosome_results_61.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

# ------------------------------------------------------------
# 6. Summarize subgroup variation
# ------------------------------------------------------------

print("\n2. CHROMOSOME COVERAGE")

print(
    "Chromosomes represented:",
    len(chromosome_results_61)
)

print(
    "Total variants:",
    chromosome_results_61["Variants"].sum()
)

print(
    "Smallest chromosome subgroup:",
    int(chromosome_results_61["Variants"].min())
)

print(
    "Largest chromosome subgroup:",
    int(chromosome_results_61["Variants"].max())
)

eligible_61 = chromosome_results_61[
    chromosome_results_61["Variants"] >= 100
]

print("\n3. SUBGROUPS WITH AT LEAST 100 VARIANTS")

print("Number of chromosomes:", len(eligible_61))

if not eligible_61.empty:
    print(
        "ROC-AUC range:",
        f"{eligible_61['ROC-AUC'].min():.4f}",
        "to",
        f"{eligible_61['ROC-AUC'].max():.4f}"
    )

    print(
        "F1 range:",
        f"{eligible_61['F1'].min():.4f}",
        "to",
        f"{eligible_61['F1'].max():.4f}"
    )

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

output_dir_61 = TABLES_DIR

chromosome_path_61 = (
    output_dir_61
    / "final_lightgbm_chromosome_performance.csv"
)

chromosome_results_61.to_csv(
    chromosome_path_61,
    index=False
)

assert chromosome_path_61.is_file()

print("\n4. SAVED FILE")
print(chromosome_path_61)

# ------------------------------------------------------------
# 8. Reproducibility
# ------------------------------------------------------------

print("\n5. REPRODUCIBILITY")
print("Final model: LightGBM")
print("Test variants:", len(chromosome_test_61))
print("Classification threshold: 0.50")
print("Model retrained: No")
print("Threshold changed: No")
print("Subgroup analysis: descriptive only")

print("\n=== Cell 61 Complete ===")

## Analysis Cell 62

In [ ]:
# ============================================================
# Cell 62 — Gene Overlap Between Development and Test Sets
# ============================================================

import numpy as np
import pandas as pd

print("=== Cell 62: Gene Overlap Audit ===")

# ------------------------------------------------------------
# 1. Validate source annotations
# ------------------------------------------------------------

key_cols_62 = ["chrom", "pos", "ref", "alt"]

assert all(c in df.columns for c in key_cols_62 + ["genename"])
assert all(c in variants_train_42.columns for c in key_cols_62)
assert all(c in variants_test_42.columns for c in key_cols_62)

def variant_key_62(frame):
    return pd.MultiIndex.from_frame(
        frame[key_cols_62].astype(str)
    )

source_keys_62 = variant_key_62(df)
train_keys_62 = variant_key_62(variants_train_42)
test_keys_62 = variant_key_62(variants_test_42)

assert train_keys_62.is_unique
assert test_keys_62.is_unique
assert len(train_keys_62.intersection(test_keys_62)) == 0

# ------------------------------------------------------------
# 2. Parse gene annotations
# ------------------------------------------------------------

def parse_genes_62(value):
    if pd.isna(value):
        return set()

    return {
        gene.strip()
        for gene in str(value).split(";")
        if gene.strip() not in {"", ".", "nan", "NA"}
    }

gene_map_62 = {}

for key, annotation in zip(
    source_keys_62,
    df["genename"]
):
    gene_map_62.setdefault(key, set()).update(
        parse_genes_62(annotation)
    )

# ------------------------------------------------------------
# 3. Identify development and test genes
# ------------------------------------------------------------

development_genes_62 = set()

for key in train_keys_62:
    development_genes_62.update(
        gene_map_62.get(key, set())
    )

test_genes_62 = set()

for key in test_keys_62:
    test_genes_62.update(
        gene_map_62.get(key, set())
    )

shared_genes_62 = (
    development_genes_62 & test_genes_62
)

# ------------------------------------------------------------
# 4. Classify test variants by gene overlap
# ------------------------------------------------------------

overlap_rows_62 = []

for key in test_keys_62:

    genes = gene_map_62.get(key, set())

    if not genes:
        status = "No gene annotation"
    elif genes & development_genes_62:
        status = "At least one gene in development"
    else:
        status = "All annotated genes unseen"

    overlap_rows_62.append({
        "chrom": key[0],
        "pos": key[1],
        "ref": key[2],
        "alt": key[3],
        "gene_count": len(genes),
        "gene_overlap_status": status
    })

gene_overlap_62 = pd.DataFrame(overlap_rows_62)

assert len(gene_overlap_62) == len(test_keys_62)

# ------------------------------------------------------------
# 5. Report overlap
# ------------------------------------------------------------

print("\n1. GENE COVERAGE")

print("Development genes:", len(development_genes_62))
print("Test genes:", len(test_genes_62))
print("Genes shared across partitions:", len(shared_genes_62))

print("\n2. TEST VARIANT GENE OVERLAP")

overlap_counts_62 = (
    gene_overlap_62["gene_overlap_status"]
    .value_counts()
)

for status, count in overlap_counts_62.items():
    percentage = 100 * count / len(gene_overlap_62)
    print(f"{status}: {count:,} ({percentage:.2f}%)")

print("\n3. SOURCE ANNOTATION COVERAGE")

missing_gene_62 = sum(
    not gene_map_62.get(key, set())
    for key in test_keys_62
)

print("Test variants without gene annotation:", missing_gene_62)
print(
    "Test variants with multiple annotated genes:",
    int((gene_overlap_62["gene_count"] > 1).sum())
)

# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

output_dir_62 = (
    TABLES_DIR
)

output_dir_62.mkdir(
    parents=True,
    exist_ok=True
)

overlap_path_62 = (
    output_dir_62 / "final_test_gene_overlap_audit.csv"
)

gene_overlap_62.to_csv(
    overlap_path_62,
    index=False
)

assert overlap_path_62.is_file()

print("\n4. SAVED FILE")
print(overlap_path_62)

print("\n5. INTERPRETATION")
print(
    "Shared genes indicate that the variant-level test "
    "split is not gene-independent. This audit does not "
    "constitute an unseen-gene validation experiment."
)

print("\n=== Cell 62 Complete ===")

## Analysis Cell 63

In [ ]:
# ============================================================
# Cell 63 — Seen-Gene vs. Unseen-Gene Test Performance
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

print("=== Cell 63: Gene-Overlap Subgroup Performance ===")

# ------------------------------------------------------------
# 1. Verify alignment of test predictions and gene audit
# ------------------------------------------------------------

key_cols_63 = ["chrom", "pos", "ref", "alt"]

predictions_63 = test_predictions_59.copy()
overlap_63 = gene_overlap_62.copy()

for frame_63 in [predictions_63, overlap_63]:
    for col_63 in key_cols_63:
        frame_63[col_63] = frame_63[col_63].astype(str)

assert len(predictions_63) == 40245
assert len(overlap_63) == 40245

assert not predictions_63.duplicated(key_cols_63).any()
assert not overlap_63.duplicated(key_cols_63).any()

subgroups_63 = predictions_63.merge(
    overlap_63[
        key_cols_63 + [
            "gene_count",
            "gene_overlap_status"
        ]
    ],
    on=key_cols_63,
    how="left",
    validate="one_to_one"
)

assert len(subgroups_63) == 40245
assert subgroups_63["gene_overlap_status"].notna().all()

# ------------------------------------------------------------
# 2. Calculate metrics within each subgroup
# ------------------------------------------------------------

rows_63 = []

for status_63, group_63 in subgroups_63.groupby(
    "gene_overlap_status",
    sort=True
):

    y_true_group_63 = group_63[
        "true_label"
    ].to_numpy(dtype=int)

    y_pred_group_63 = group_63[
        "predicted_label"
    ].to_numpy(dtype=int)

    y_prob_group_63 = group_63[
        "pathogenic_probability"
    ].to_numpy(dtype=float)

    tn_63, fp_63, fn_63, tp_63 = confusion_matrix(
        y_true_group_63,
        y_pred_group_63,
        labels=[0, 1]
    ).ravel()

    has_both_classes_63 = (
        len(np.unique(y_true_group_63)) == 2
    )

    rows_63.append({
        "Gene overlap group": status_63,
        "Variants": len(group_63),
        "Benign": int((y_true_group_63 == 0).sum()),
        "Pathogenic": int((y_true_group_63 == 1).sum()),
        "Pathogenic prevalence": y_true_group_63.mean(),
        "Accuracy": accuracy_score(
            y_true_group_63,
            y_pred_group_63
        ),
        "Balanced accuracy": (
            balanced_accuracy_score(
                y_true_group_63,
                y_pred_group_63
            )
            if has_both_classes_63
            else np.nan
        ),
        "Precision": precision_score(
            y_true_group_63,
            y_pred_group_63,
            zero_division=0
        ),
        "Sensitivity": (
            tp_63 / (tp_63 + fn_63)
            if (tp_63 + fn_63) > 0
            else np.nan
        ),
        "Specificity": (
            tn_63 / (tn_63 + fp_63)
            if (tn_63 + fp_63) > 0
            else np.nan
        ),
        "F1": f1_score(
            y_true_group_63,
            y_pred_group_63,
            zero_division=0
        ),
        "ROC-AUC": (
            roc_auc_score(
                y_true_group_63,
                y_prob_group_63
            )
            if has_both_classes_63
            else np.nan
        ),
        "PR-AUC": (
            average_precision_score(
                y_true_group_63,
                y_prob_group_63
            )
            if has_both_classes_63
            else np.nan
        ),
        "True negatives": int(tn_63),
        "False positives": int(fp_63),
        "False negatives": int(fn_63),
        "True positives": int(tp_63)
    })

gene_subgroup_results_63 = pd.DataFrame(rows_63)

# ------------------------------------------------------------
# 3. Verify group totals
# ------------------------------------------------------------

assert gene_subgroup_results_63["Variants"].sum() == 40245
assert gene_subgroup_results_63["Benign"].sum() == 27480
assert gene_subgroup_results_63["Pathogenic"].sum() == 12765

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

print("\n1. GENE-OVERLAP SUBGROUP METRICS")

display_cols_63 = [
    "Gene overlap group",
    "Variants",
    "Benign",
    "Pathogenic",
    "Pathogenic prevalence",
    "Accuracy",
    "Balanced accuracy",
    "Sensitivity",
    "Specificity",
    "F1",
    "ROC-AUC",
    "PR-AUC"
]

print(
    gene_subgroup_results_63[
        display_cols_63
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

print("\n2. SUBGROUP CONFUSION COUNTS")

print(
    gene_subgroup_results_63[
        [
            "Gene overlap group",
            "True negatives",
            "False positives",
            "False negatives",
            "True positives"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 5. Save subgroup results
# ------------------------------------------------------------

output_dir_63 = TABLES_DIR
output_dir_63.mkdir(parents=True, exist_ok=True)

results_path_63 = (
    output_dir_63
    / "final_lightgbm_seen_vs_unseen_gene_performance.csv"
)

gene_subgroup_results_63.to_csv(
    results_path_63,
    index=False
)

assert results_path_63.is_file()

print("\n3. SAVED FILE")
print(results_path_63)

print("\n4. INTERPRETATION")
print(
    "This is a post hoc descriptive comparison within "
    "the original variant-level test partition."
)
print(
    "Unseen-gene subgroup membership was not used "
    "to construct a gene-disjoint training/test split."
)
print("Model retrained: No")
print("Classification threshold changed: No")

print("\n=== Cell 63 Complete ===")

## Analysis Cell 64

In [ ]:
# ============================================================
# Cell 64 — Reconstructed Figure 2: Six-Model ROC Curves
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score

print("=== Cell 64: Reconstructed Figure 2 — ROC Curves ===")

# 1. Use the common validation partition.
y_64 = np.asarray(y_val_46, dtype=int)

models_64 = {
    "Logistic Regression": (logreg_48, X_val_scaled_47),
    "Random Forest": (rf_49, X_val_imputed_47),
    "XGBoost": (xgb_50, X_val_imputed_47),
    "LightGBM": (lgbm_51, X_val_imputed_47),
    "Linear SVM": (svm_52, X_val_scaled_47),
    "KNN": (knn_53, X_val_scaled_47),
}

# 2. Calculate ROC curves and AUC values.
roc_data_64 = {}

for name, (model, features) in models_64.items():
    probabilities = model.predict_proba(features)[:, 1]

    fpr, tpr, _ = roc_curve(y_64, probabilities)
    auc = roc_auc_score(y_64, probabilities)

    roc_data_64[name] = {
        "fpr": fpr,
        "tpr": tpr,
        "auc": auc
    }

# 3. Create publication-quality figure.
fig_64, ax_64 = plt.subplots(figsize=(8, 7))

for name, values in roc_data_64.items():
    ax_64.plot(
        values["fpr"],
        values["tpr"],
        linewidth=2,
        label=f"{name} (AUC = {values['auc']:.3f})"
    )

ax_64.plot(
    [0, 1], [0, 1],
    linestyle="--",
    color="gray",
    linewidth=1.2,
    label="Random classifier"
)

ax_64.set(
    xlabel="False Positive Rate (1 − Specificity)",
    ylabel="True Positive Rate (Sensitivity)",
    title="ROC Curves of Six Machine Learning Models",
    xlim=(0, 1),
    ylim=(0, 1.01)
)

ax_64.legend(loc="lower right", fontsize=9)
ax_64.grid(alpha=0.2)

fig_64.tight_layout()

# 4. Save PNG and vector PDF.
figure_dir_64 = FIGURES_DIR
figure_dir_64.mkdir(parents=True, exist_ok=True)

png_path_64 = figure_dir_64 / "Figure_2_ROC_curves_reconstructed.png"
pdf_path_64 = figure_dir_64 / "Figure_2_ROC_curves_reconstructed.pdf"

fig_64.savefig(png_path_64, dpi=300, bbox_inches="tight")
fig_64.savefig(pdf_path_64, bbox_inches="tight")

plt.show()

# 5. Report and verify.
print("\nValidation variants:", len(y_64))

for name, values in roc_data_64.items():
    print(f"{name}: ROC-AUC = {values['auc']:.5f}")

assert png_path_64.is_file()
assert pdf_path_64.is_file()

print("\nSaved PNG:", png_path_64)
print("Saved PDF:", pdf_path_64)
print("\nFigure label: Reconstructed Figure 2")
print("Evaluation partition: Validation")
print("Original published results reproduced exactly: No")

print("\n=== Cell 64 Complete ===")

## Analysis Cell 65

In [ ]:
# ============================================================
# Cell 65 — Reconstructed Precision–Recall Curves
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

print("=== Cell 65: Reconstructed Precision–Recall Curves ===")

# ------------------------------------------------------------
# 1. Use the same validation data and models as Cell 64
# ------------------------------------------------------------

y_65 = np.asarray(y_val_46, dtype=int)

assert len(y_65) == 32196
assert set(np.unique(y_65)) == {0, 1}

models_65 = {
    "Logistic Regression": (logreg_48, X_val_scaled_47),
    "Random Forest": (rf_49, X_val_imputed_47),
    "XGBoost": (xgb_50, X_val_imputed_47),
    "LightGBM": (lgbm_51, X_val_imputed_47),
    "Linear SVM": (svm_52, X_val_scaled_47),
    "KNN": (knn_53, X_val_scaled_47)
}

# ------------------------------------------------------------
# 2. Calculate precision–recall curves
# ------------------------------------------------------------

pr_data_65 = {}

for name_65, (model_65, features_65) in models_65.items():

    probabilities_65 = model_65.predict_proba(
        features_65
    )[:, 1]

    precision_65, recall_65, _ = precision_recall_curve(
        y_65,
        probabilities_65
    )

    ap_65 = average_precision_score(
        y_65,
        probabilities_65
    )

    pr_data_65[name_65] = {
        "precision": precision_65,
        "recall": recall_65,
        "average_precision": ap_65
    }

# ------------------------------------------------------------
# 3. Create publication-quality figure
# ------------------------------------------------------------

fig_65, ax_65 = plt.subplots(figsize=(8, 7))

for name_65, values_65 in pr_data_65.items():

    ax_65.plot(
        values_65["recall"],
        values_65["precision"],
        linewidth=2,
        label=(
            f"{name_65} "
            f"(AP = {values_65['average_precision']:.3f})"
        )
    )

baseline_65 = float(y_65.mean())

ax_65.axhline(
    baseline_65,
    linestyle="--",
    color="gray",
    linewidth=1.2,
    label=f"Class prevalence ({baseline_65:.3f})"
)

ax_65.set(
    xlabel="Recall (Sensitivity)",
    ylabel="Precision",
    title="Precision–Recall Curves of Six Machine Learning Models",
    xlim=(0, 1),
    ylim=(0, 1.01)
)

ax_65.legend(loc="lower left", fontsize=9)
ax_65.grid(alpha=0.2)

fig_65.tight_layout()

# ------------------------------------------------------------
# 4. Save PNG and vector PDF
# ------------------------------------------------------------

figure_dir_65 = FIGURES_DIR

figure_dir_65.mkdir(parents=True, exist_ok=True)

png_path_65 = (
    figure_dir_65
    / "Figure_3_PR_curves_reconstructed.png"
)

pdf_path_65 = (
    figure_dir_65
    / "Figure_3_PR_curves_reconstructed.pdf"
)

fig_65.savefig(
    png_path_65,
    dpi=300,
    bbox_inches="tight"
)

fig_65.savefig(
    pdf_path_65,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 5. Report results
# ------------------------------------------------------------

print("\n1. VALIDATION AVERAGE PRECISION")

for name_65, values_65 in pr_data_65.items():
    print(
        f"{name_65}: "
        f"AP = {values_65['average_precision']:.5f}"
    )

print("\n2. SAVED FIGURES")

assert png_path_65.is_file()
assert pdf_path_65.is_file()

print("PNG:", png_path_65)
print("PDF:", pdf_path_65)

print("\n3. REPRODUCIBILITY")
print("Validation variants:", len(y_65))
print("Pathogenic prevalence:", f"{baseline_65:.5f}")
print("Models retrained: No")
print("Reserved test set used: No")
print("Published numerical results reproduced exactly: No")

print("\n=== Cell 65 Complete ===")

## Analysis Cell 66

In [ ]:
# ============================================================
# Cell 66 — Article-Style SHAP Beeswarm (Figure 5)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import shap

print("=== Cell 66: SHAP Beeswarm — Article Style ===")

# 1. Reproducible sample from development data
X_raw_66 = X_train_42.sample(
    n=2000,
    random_state=42
)

feature_names_66 = X_train_42.columns.tolist()

# 2. Apply fitted imputation (no test data)
X_shap_66 = pd.DataFrame(
    final_lgbm_57.named_steps["imputer"].transform(X_raw_66),
    columns=feature_names_66,
    index=X_raw_66.index
)

# 3. Calculate LightGBM SHAP values
explainer_66 = shap.TreeExplainer(
    final_lgbm_57.named_steps["classifier"]
)

values_66 = explainer_66.shap_values(X_shap_66)

if isinstance(values_66, list):
    values_66 = values_66[-1]

values_66 = np.asarray(values_66)

if values_66.ndim == 3:
    values_66 = values_66[:, :, 1]

assert values_66.shape == X_shap_66.shape

# 4. Create SHAP Explanation object
explanation_66 = shap.Explanation(
    values=values_66,
    data=X_shap_66.to_numpy(),
    feature_names=feature_names_66
)

# 5. Article-style beeswarm
plt.figure(figsize=(12, 8))

shap.plots.beeswarm(
    explanation_66,
    max_display=13,
    show=False,
    plot_size=None,
    color=plt.get_cmap("cool")
)

fig_66 = plt.gcf()
fig_66.set_size_inches(12, 8)

ax_66 = fig_66.axes[0]
ax_66.set_title(
    "SHAP Summary – LightGBM",
    fontsize=19,
    pad=18
)

ax_66.set_xlabel(
    "SHAP value (impact on model output)",
    fontsize=14
)

ax_66.tick_params(axis="both", labelsize=12)

fig_66.tight_layout()

# 6. Save publication-quality figures
figure_dir_66 = FIGURES_DIR

figure_dir_66.mkdir(parents=True, exist_ok=True)

png_path_66 = (
    figure_dir_66
    / "Figure_5_SHAP_beeswarm_reconstructed.png"
)

pdf_path_66 = (
    figure_dir_66
    / "Figure_5_SHAP_beeswarm_reconstructed.pdf"
)

fig_66.savefig(
    png_path_66,
    dpi=300,
    bbox_inches="tight"
)

fig_66.savefig(
    pdf_path_66,
    bbox_inches="tight"
)

plt.show()

# 7. Feature-importance table
importance_66 = pd.DataFrame({
    "Predictor": feature_names_66,
    "Mean absolute SHAP": np.abs(values_66).mean(axis=0)
}).sort_values(
    "Mean absolute SHAP",
    ascending=False
)

importance_path_66 = (
    figure_dir_66.parent
    / "shap_feature_importance.csv"
)

importance_66.to_csv(
    importance_path_66,
    index=False
)

print("\n1. SHAP FEATURE IMPORTANCE")
print(
    importance_66.to_string(
        index=False,
        float_format=lambda x: f"{x:.5f}"
    )
)

print("\n2. SAVED FIGURES")
print("PNG:", png_path_66)
print("PDF:", pdf_path_66)
print("Importance CSV:", importance_path_66)

print("\n3. REPRODUCIBILITY")
print("SHAP version:", shap.__version__)
print("Explained development variants:", len(X_shap_66))
print("Predictors:", len(feature_names_66))
print("Reserved test set used: No")
print("Original 28-predictor model reproduced: No")

print("\n=== Cell 66 Complete ===")

## Analysis Cell 67

In [ ]:
# ============================================================
# Cell 67 — SHAP Beeswarm: Random Forest, XGBoost, LightGBM
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from pathlib import Path

print("=== Cell 67: SHAP Beeswarm Plots — Top Three Models ===")

# ------------------------------------------------------------
# 1. Use the same 2,000 development variants for all models
# ------------------------------------------------------------

X_raw_67 = X_train_42.sample(
    n=2000,
    random_state=42
)

feature_names_67 = X_train_42.columns.tolist()

# All three models were trained on median-imputed features.
X_shap_67 = pd.DataFrame(
    final_lgbm_57.named_steps["imputer"].transform(X_raw_67),
    columns=feature_names_67,
    index=X_raw_67.index
)

assert X_shap_67.shape == (2000, 13)

# ------------------------------------------------------------
# 2. Define the three trained models
# ------------------------------------------------------------

models_67 = {
    "Random Forest": rf_49,
    "XGBoost": xgb_50,
    "LightGBM": final_lgbm_57.named_steps["classifier"]
}

figure_numbers_67 = {
    "Random Forest": 4,
    "XGBoost": 5,
    "LightGBM": 6
}

figure_dir_67 = FIGURES_DIR

figure_dir_67.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Generate SHAP beeswarm for each model
# ------------------------------------------------------------

shap_importance_tables_67 = []

for model_name_67, model_67 in models_67.items():

    print(f"\nProcessing {model_name_67}...")

    explainer_67 = shap.TreeExplainer(model_67)

    shap_values_67 = explainer_67.shap_values(
        X_shap_67
    )

    if isinstance(shap_values_67, list):
        shap_values_67 = shap_values_67[-1]

    shap_values_67 = np.asarray(shap_values_67)

    # Some SHAP versions return one matrix per class.
    if shap_values_67.ndim == 3:
        shap_values_67 = shap_values_67[:, :, 1]

    assert shap_values_67.shape == X_shap_67.shape

    explanation_67 = shap.Explanation(
        values=shap_values_67,
        data=X_shap_67.to_numpy(),
        feature_names=feature_names_67
    )

    # --------------------------------------------------------
    # 4. Article-style beeswarm
    # --------------------------------------------------------

    plt.figure(figsize=(12, 8))

    shap.plots.beeswarm(
        explanation_67,
        max_display=13,
        show=False,
        plot_size=None,
        color=plt.get_cmap("cool")
    )

    fig_67 = plt.gcf()
    fig_67.set_size_inches(12, 8)

    ax_67 = fig_67.axes[0]

    ax_67.set_title(
        f"SHAP Summary – {model_name_67}",
        fontsize=19,
        pad=18
    )

    ax_67.set_xlabel(
        "SHAP value (impact on model output)",
        fontsize=14
    )

    ax_67.tick_params(
        axis="both",
        labelsize=12
    )

    fig_67.tight_layout()

    # --------------------------------------------------------
    # 5. Save figure as PNG and PDF
    # --------------------------------------------------------

    figure_number_67 = figure_numbers_67[model_name_67]

    safe_name_67 = model_name_67.replace(" ", "_")

    png_path_67 = (
        figure_dir_67
        / f"Figure_{figure_number_67}_SHAP_{safe_name_67}.png"
    )

    pdf_path_67 = (
        figure_dir_67
        / f"Figure_{figure_number_67}_SHAP_{safe_name_67}.pdf"
    )

    fig_67.savefig(
        png_path_67,
        dpi=300,
        bbox_inches="tight"
    )

    fig_67.savefig(
        pdf_path_67,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig_67)

    # --------------------------------------------------------
    # 6. Save feature-importance values
    # --------------------------------------------------------

    importance_67 = pd.DataFrame({
        "Model": model_name_67,
        "Predictor": feature_names_67,
        "Mean absolute SHAP": np.abs(
            shap_values_67
        ).mean(axis=0)
    })

    shap_importance_tables_67.append(importance_67)

    print("Saved PNG:", png_path_67)
    print("Saved PDF:", pdf_path_67)

# ------------------------------------------------------------
# 7. Combine importance results
# ------------------------------------------------------------

all_shap_importance_67 = pd.concat(
    shap_importance_tables_67,
    ignore_index=True
)

importance_path_67 = (
    figure_dir_67.parent
    / "shap_top_three_models_importance.csv"
)

all_shap_importance_67.to_csv(
    importance_path_67,
    index=False
)

print("\n1. TOP FIVE FEATURES PER MODEL")

for model_name_67 in models_67:

    top_features_67 = (
        all_shap_importance_67[
            all_shap_importance_67["Model"] == model_name_67
        ]
        .sort_values(
            "Mean absolute SHAP",
            ascending=False
        )
        .head(5)
    )

    print(f"\n{model_name_67}")
    print(
        top_features_67[
            ["Predictor", "Mean absolute SHAP"]
        ].to_string(
            index=False,
            float_format=lambda x: f"{x:.5f}"
        )
    )

print("\n2. SAVED IMPORTANCE TABLE")
print(importance_path_67)

print("\n3. REPRODUCIBILITY")
print("SHAP version:", shap.__version__)
print("Models:", ", ".join(models_67.keys()))
print("Explained development variants:", len(X_shap_67))
print("Predictors:", len(feature_names_67))
print("Models retrained: No")
print("Reserved test set used: No")
print(
    "Note: SHAP values from different model types "
    "may use different output scales."
)

print("\n=== Cell 67 Complete ===")

## Analysis Cell 68

In [ ]:
# ============================================================
# Cell 68 — Reconstructed Article Table 1
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("=== Cell 68: Reconstructed Article Table 1 ===")

# 1. Use actual six-model validation results.
source_68 = validation_results_53.copy()

model_order_68 = [
    "Logistic Regression",
    "Random Forest",
    "XGBoost",
    "LightGBM",
    "KNN (k=5)",
    "Linear SVM (calibrated)"
]

display_names_68 = {
    "Logistic Regression": "Logistic Regression",
    "Random Forest": "Random Forest",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "KNN (k=5)": "KNN",
    "Linear SVM (calibrated)": "SVM (Linear)"
}

source_68 = source_68.set_index("Model")

assert all(
    model in source_68.index
    for model in model_order_68
)

table_68 = source_68.loc[
    model_order_68,
    ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
].copy()

table_68 = table_68.rename(
    columns={
        "F1": "F1-Score"
    }
)

table_68.insert(
    0,
    "Model",
    [display_names_68[m] for m in model_order_68]
)

table_68 = table_68.reset_index(drop=True)

# 2. Save actual numeric results.
output_dir_68 = TABLES_DIR
output_dir_68.mkdir(parents=True, exist_ok=True)

csv_path_68 = (
    output_dir_68 / "Table_1_validation_performance.csv"
)

table_68.to_csv(csv_path_68, index=False)

# 3. Format the displayed numbers.
formatted_68 = table_68.copy()

for column_68 in [
    "Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"
]:
    formatted_68[column_68] = formatted_68[column_68].map(
        lambda value: f"{value:.4f}"
    )

# 4. Create an article-style table.
fig_68, ax_68 = plt.subplots(figsize=(14, 5.3))
ax_68.axis("off")

fig_68.text(
    0.06, 0.92,
    "Table 1.",
    fontsize=17,
    fontweight="bold",
    color="#354766"
)

fig_68.text(
    0.06, 0.84,
    "Performance metrics of evaluated classifiers "
    "on the validation partition.",
    fontsize=13,
    color="#354766"
)

fig_68.text(
    0.06, 0.78,
    "Single stratified validation split (n = 32,196); "
    "not five-fold cross-validation.",
    fontsize=10,
    color="#59677A"
)

column_labels_68 = formatted_68.columns.tolist()

cell_text_68 = formatted_68.values.tolist()

table_plot_68 = ax_68.table(
    cellText=cell_text_68,
    colLabels=column_labels_68,
    cellLoc="left",
    colLoc="left",
    bbox=[0.06, 0.10, 0.88, 0.59],
    colWidths=[0.27, 0.145, 0.145, 0.145, 0.145, 0.15]
)

table_plot_68.auto_set_font_size(False)
table_plot_68.set_fontsize(11)

# Remove vertical borders and use only horizontal rules.
for (row_68, col_68), cell_68 in table_plot_68.get_celld().items():

    cell_68.set_facecolor("white")
    cell_68.set_edgecolor("#29364A")
    cell_68.set_linewidth(0)
    cell_68.visible_edges = ""

    if row_68 == 0:
        cell_68.set_text_props(
            weight="bold",
            color="#172235"
        )
        cell_68.visible_edges = "TB"
        cell_68.set_linewidth(0.8)
    else:
        cell_68.set_text_props(color="#354155")

    if row_68 == len(formatted_68):
        cell_68.visible_edges = "B"
        cell_68.set_linewidth(0.8)

# 5. Save publication-quality versions.
figure_dir_68 = FIGURES_DIR
figure_dir_68.mkdir(parents=True, exist_ok=True)

png_path_68 = (
    figure_dir_68 / "Table_1_validation_performance.png"
)

pdf_path_68 = (
    figure_dir_68 / "Table_1_validation_performance.pdf"
)

fig_68.savefig(
    png_path_68,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig_68.savefig(
    pdf_path_68,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

# 6. Report.
print("\n1. TABLE 1 — VALIDATION PERFORMANCE")

print(
    formatted_68.to_string(index=False)
)

print("\n2. SAVED FILES")
print("CSV:", csv_path_68)
print("PNG:", png_path_68)
print("PDF:", pdf_path_68)

print("\n3. REPRODUCIBILITY")
print("Models:", len(table_68))
print("Validation variants:", len(y_val_46))
print("Evaluation: single validation split")
print("Five-fold cross-validation performed: No")
print("Standard deviations estimated: No")
print("Model retraining: No")
print("Reserved test set used: No")

print("\n=== Cell 68 Complete ===")

## Analysis Cell 69

In [ ]:
# ============================================================
# Cell 69 — Reconstructed Article Table 2: Top 10 SHAP Features
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("=== Cell 69: Reconstructed Article Table 2 ===")

# ------------------------------------------------------------
# 1. Use final LightGBM SHAP results from Cell 66
# ------------------------------------------------------------

assert "importance_66" in globals()

assert all(
    column in importance_66.columns
    for column in ["Predictor", "Mean absolute SHAP"]
)

top10_69 = (
    importance_66[
        ["Predictor", "Mean absolute SHAP"]
    ]
    .sort_values(
        "Mean absolute SHAP",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

assert len(top10_69) == 10

# ------------------------------------------------------------
# 2. Add feature interpretations
# ------------------------------------------------------------

interpretations_69 = {
    "MutationTaster_score": "Predicted functional impact of variants",
    "CADD_phred": "Integrated variant deleteriousness score",
    "MetaSVM_score": "Ensemble pathogenicity prediction",
    "MetaLR_score": "Ensemble pathogenicity prediction",
    "VEST4_score": "Missense variant effect prediction",
    "fathmm-XF_coding_score": "Coding variant functional impact",
    "REVEL_score": "Ensemble missense pathogenicity score",
    "MutationAssessor_score": "Protein functional impact prediction",
    "SIFT_score": "Amino acid substitution tolerance",
    "DANN_score": "Neural-network deleteriousness score",
    "Polyphen2_HVAR_score": "Predicted protein substitution impact",
    "PROVEAN_score": "Protein sequence variation impact",
    "Polyphen2_HDIV_score": "Predicted protein substitution impact"
}

table_69 = pd.DataFrame({
    "Rank": np.arange(1, 11),
    "Feature Name": top10_69["Predictor"],
    "Mean SHAP Value": top10_69["Mean absolute SHAP"],
    "Contribution Interpretation": (
        top10_69["Predictor"].map(interpretations_69)
    )
})

assert table_69["Contribution Interpretation"].notna().all()

# ------------------------------------------------------------
# 3. Save numeric table
# ------------------------------------------------------------

output_dir_69 = TABLES_DIR
output_dir_69.mkdir(parents=True, exist_ok=True)

csv_path_69 = (
    output_dir_69 / "Table_2_top10_SHAP_predictors.csv"
)

table_69.to_csv(csv_path_69, index=False)

# ------------------------------------------------------------
# 4. Format table for publication
# ------------------------------------------------------------

formatted_69 = table_69.copy()

formatted_69["Mean SHAP Value"] = (
    formatted_69["Mean SHAP Value"]
    .map(lambda value: f"{value:.3f}")
)

fig_69, ax_69 = plt.subplots(figsize=(16, 7.6))
ax_69.axis("off")

fig_69.text(
    0.045, 0.94,
    "Table 2.",
    fontsize=19,
    fontweight="bold",
    color="#354766"
)

fig_69.text(
    0.045, 0.885,
    "Top 10 most informative predictors ranked by mean absolute "
    "SHAP value from the final LightGBM model.",
    fontsize=13,
    color="#354766"
)

# ------------------------------------------------------------
# 5. Create article-style table
# ------------------------------------------------------------

plot_table_69 = ax_69.table(
    cellText=formatted_69.values.tolist(),
    colLabels=formatted_69.columns.tolist(),
    cellLoc="left",
    colLoc="left",
    bbox=[0.045, 0.085, 0.91, 0.72],
    colWidths=[0.07, 0.29, 0.17, 0.47]
)

plot_table_69.auto_set_font_size(False)
plot_table_69.set_fontsize(11)

for (row_69, col_69), cell_69 in plot_table_69.get_celld().items():

    cell_69.set_facecolor("white")
    cell_69.set_edgecolor("#29364A")
    cell_69.set_linewidth(0)
    cell_69.visible_edges = ""

    if row_69 == 0:
        cell_69.set_text_props(
            weight="bold",
            color="#172235"
        )
        cell_69.visible_edges = "TB"
        cell_69.set_linewidth(0.8)

    else:
        cell_69.set_text_props(color="#354155")

    if row_69 == len(formatted_69):
        cell_69.visible_edges = "B"
        cell_69.set_linewidth(0.8)

fig_69.tight_layout()

# ------------------------------------------------------------
# 6. Save PNG and PDF
# ------------------------------------------------------------

figure_dir_69 = FIGURES_DIR
figure_dir_69.mkdir(parents=True, exist_ok=True)

png_path_69 = (
    figure_dir_69 / "Table_2_top10_SHAP_predictors.png"
)

pdf_path_69 = (
    figure_dir_69 / "Table_2_top10_SHAP_predictors.pdf"
)

fig_69.savefig(
    png_path_69,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig_69.savefig(
    pdf_path_69,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

# ------------------------------------------------------------
# 7. Report results
# ------------------------------------------------------------

print("\n1. TABLE 2 — TOP 10 SHAP PREDICTORS")

print(
    table_69.to_string(
        index=False,
        float_format=lambda value: f"{value:.5f}"
    )
)

print("\n2. SAVED FILES")
print("CSV:", csv_path_69)
print("PNG:", png_path_69)
print("PDF:", pdf_path_69)

print("\n3. REPRODUCIBILITY")
print("Model: Final LightGBM")
print("SHAP sample: 2,000 development variants")
print("Features ranked: 10 of 13")
print("Ranking criterion: mean absolute SHAP value")
print("Model retrained: No")
print("SHAP recomputed: No")
print("Reserved test set used: No")
print("Published numerical results reproduced exactly: No")

print("\n=== Cell 69 Complete ===")

## Analysis Cell 70

In [ ]:
# ============================================================
# Cell 70 — Reconstructed Article Table 3: Dataset Summary
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("=== Cell 70: Reconstructed Article Table 3 ===")

# ------------------------------------------------------------
# 1. Verify dataset counts
# ------------------------------------------------------------

assert len(df) == 203103
assert len(y_40) == 201225
assert len(y_train_42) == 160980
assert len(y_test_42) == 40245

assert len(y_train_42) + len(y_test_42) == len(y_40)

# ------------------------------------------------------------
# 2. Construct dataset-summary table
# ------------------------------------------------------------

table_70 = pd.DataFrame([
    {
        "Dataset Component": "ClinVar labeled GRCh37 SNVs",
        "Count": 1354204,
        "Notes": "Unique benign/pathogenic candidates"
    },
    {
        "Dataset Component": "Matched missense annotation records",
        "Count": len(df),
        "Notes": "Conservative missense filter; duplicates retained"
    },
    {
        "Dataset Component": "Unique annotated missense variants",
        "Count": len(y_40),
        "Notes": "After variant-level deduplication"
    },
    {
        "Dataset Component": "Development set (80%)",
        "Count": len(y_train_42),
        "Notes": "Stratified; includes fitting and validation subsets"
    },
    {
        "Dataset Component": "Test set (20%)",
        "Count": len(y_test_42),
        "Notes": "Stratified; reserved for final evaluation"
    },
    {
        "Dataset Component": "Complete cases (13 predictors)",
        "Count": 86960,
        "Notes": "Descriptive only; median imputation used for modeling"
    }
])

assert len(table_70) == 6

# ------------------------------------------------------------
# 3. Save numeric CSV
# ------------------------------------------------------------

output_dir_70 = TABLES_DIR
output_dir_70.mkdir(parents=True, exist_ok=True)

csv_path_70 = output_dir_70 / "Table_3_dataset_summary.csv"

table_70.to_csv(csv_path_70, index=False)

# ------------------------------------------------------------
# 4. Format table for publication
# ------------------------------------------------------------

formatted_70 = table_70.copy()

formatted_70["Count"] = formatted_70["Count"].map(
    lambda value: f"{value:,}"
)

fig_70, ax_70 = plt.subplots(figsize=(16, 6.8))
ax_70.axis("off")

fig_70.text(
    0.045, 0.94,
    "Table 3.",
    fontsize=19,
    fontweight="bold",
    color="#354766"
)

fig_70.text(
    0.045, 0.875,
    "Summary of ClinVar-derived datasets used for model "
    "development and evaluation,",
    fontsize=13,
    color="#354766"
)

fig_70.text(
    0.045, 0.83,
    "including filtering and preprocessing steps "
    "applied prior to machine-learning analysis.",
    fontsize=13,
    color="#354766"
)

# ------------------------------------------------------------
# 5. Create article-style table
# ------------------------------------------------------------

plot_table_70 = ax_70.table(
    cellText=formatted_70.values.tolist(),
    colLabels=formatted_70.columns.tolist(),
    cellLoc="center",
    colLoc="center",
    bbox=[0.045, 0.09, 0.91, 0.64],
    colWidths=[0.36, 0.14, 0.50]
)

plot_table_70.auto_set_font_size(False)
plot_table_70.set_fontsize(11)

for (row_70, col_70), cell_70 in plot_table_70.get_celld().items():

    cell_70.set_facecolor("white")
    cell_70.set_edgecolor("#29364A")
    cell_70.set_linewidth(0)
    cell_70.visible_edges = ""

    if row_70 == 0:
        cell_70.set_text_props(
            weight="bold",
            color="#172235"
        )
        cell_70.visible_edges = "TB"
        cell_70.set_linewidth(0.8)

    else:
        cell_70.set_text_props(color="#354155")

    if row_70 == len(formatted_70):
        cell_70.visible_edges = "B"
        cell_70.set_linewidth(0.8)

fig_70.tight_layout()

# ------------------------------------------------------------
# 6. Save PNG and PDF
# ------------------------------------------------------------

figure_dir_70 = FIGURES_DIR
figure_dir_70.mkdir(parents=True, exist_ok=True)

png_path_70 = (
    figure_dir_70 / "Table_3_dataset_summary.png"
)

pdf_path_70 = (
    figure_dir_70 / "Table_3_dataset_summary.pdf"
)

fig_70.savefig(
    png_path_70,
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig_70.savefig(
    pdf_path_70,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print("\n1. TABLE 3 — DATASET SUMMARY")

print(
    table_70.to_string(
        index=False
    )
)

print("\n2. SAVED FILES")
print("CSV:", csv_path_70)
print("PNG:", png_path_70)
print("PDF:", pdf_path_70)

print("\n3. REPRODUCIBILITY")
print("Source: reconstructed ClinVar + dbNSFP dataset")
print("Final unique variants:", len(y_40))
print("Development variants:", len(y_train_42))
print("Test variants:", len(y_test_42))
print("Predictors:", X_40.shape[1])
print("Published numerical counts reproduced exactly: No")

print("\n=== Cell 70 Complete ===")